# ML-Driven Screening and QSPR Modeling of Bio-Based Green Solvents to Replace Hazardous Polar Aprotic Solvents

**Problem.** DMF, NMP and DMAc are reprotoxic (GHS H360) and are restricted / SVHC-listed under EU REACH, yet they are still the workhorse solvents for amide coupling, SNAr, cross-coupling, polymer processing and membrane casting. A replacement must keep their *solvating profile* (high dipolarity/polarizability π\*, strong H-bond basicity β, no H-bond acidity α, Hansen δD/δP/δH), stay liquid and low-volatility in the process window, **and** be safer and bio-derived.

**What this notebook does (end to end):**

| # | Section | Output |
|---|---------|--------|
| 1 | Setup | environment, seeds, config |
| 2 | Data | solvent library, RDKit curation, audit of labels |
| 3 | Featurization | RDKit 2D, MACCS, Morgan counts, optional QM (ORCA) descriptors |
| 4 | QSPR benchmark | 7 models x 3 feature sets, **nested repeated CV**, baseline comparison |
| 5 | Rigor | y-scrambling, cluster-split CV, applicability domain (Williams), SHAP |
| 6 | Screening | property prediction with uncertainty, distance to DMF/NMP/DMAc (Hansen Ra + Kamlet-Taft) |
| 7 | Ranking | multi-objective: performance, hazard, bio-origin, process window; TOPSIS + Pareto |
| 8 | Robustness | Monte-Carlo over **weights and prediction uncertainty** -> rank stability |
| 9 | Validation | blind retrospective test against known DMF replacements |
| 10 | Extras | binary-blend design, ORCA input generator, export for SI / Zenodo |

> ### READ FIRST: data status
> The built-in table is a **102-solvent curated library**. Kamlet-Taft (alpha/beta/pi*), Hansen (dD/dP/dH), dielectric constant and boiling point were taken from the Stenutz *Tables for Chemistry* web pages (see the `source` column; Hansen values converted from (cal/mL)^0.5 to MPa^0.5). **Blank = no value retrieved**, so those solvents (Cyrene, NBP, PolarClean, DMI, CPME, most levulinates, etc.) are exactly the ones the QSPR models predict. Flash points are empty (no sourced values), so that model is skipped.
> **Before publishing:** (1) replace the secondary web source with the primary literature (Jessop 2012, Marcus, Hansen handbook, CRC), (2) verify the remaining `cmr` flags and all `bio_origin` flags (only rows marked 'cmr VERIFIED' in `source` were checked), (3) add labelled bio-based solvents, which is where model coverage is thinnest. To use your own CSV set `USE_UPLOADED_CSV=True`.

**Run:** `Runtime -> Run all`. FAST mode (default) takes a few minutes; set `FAST=False` in the config cell for the final paper run.

## What makes this paper impactful (and what the notebook already covers)

| Lever reviewers reward | How it is covered here | What **you** still need to add |
|---|---|---|
| Real, urgent problem + regulatory hook | DMF/NMP/DMAc restriction framing | Cite REACH/ECHA documents in the intro |
| Large, curated, provenance-tracked dataset | `source` column, RDKit curation, InChIKey dedup | Expand to 150-300+ solvents from Jessop 2012 (Kamlet-Taft), Hansen/HSPiP, Marcus, CHEM21 guide, DIPPR/PubChem |
| Honest benchmarking | Mean-predictor baseline, nested CV, 3 feature sets, 7 model classes | Add GNN (e.g. Chemprop) only if N is large enough |
| Generalization, not memorization | y-scrambling, **cluster-split CV**, applicability domain | Report the hard (cluster) numbers in the abstract |
| OECD QSAR principles | defined endpoint, algorithm, AD, validation, mechanistic interpretation (SHAP) | A short OECD-compliance table in the SI |
| Uncertainty-aware decisions | CV-residual intervals; Monte-Carlo ranking | State P(top-3) not just a rank |
| Beyond similarity: multi-objective | Performance + hazard + bio-origin + process window, Pareto + TOPSIS, weight sensitivity | Defend weights; add LCA/price if available |
| Blind retrospective validation | Held-out-by-CV recovery of known replacements | Extend the positive list from literature |
| **Experimental validation (biggest jump in impact)** | ORCA input generator; blend design | Measure KT parameters for 3-5 top candidates, and test one application (polymer solubility / membrane casting, amide coupling, SPPS resin swelling) |
| Reproducibility | seeds, env dump, model card, outputs zip | Push code+data to GitHub, mint a Zenodo DOI |

**Rule of thumb:** the notebook gives you a rigorous computational study. Journals in the Green Chemistry / ACS Sustainable Chem. Eng. / J. Chem. Inf. Model. / Digital Discovery tier expect either *new data* (measured parameters or application tests) or a *large new curated dataset*. Plan for at least one of those.

## 1. Setup

In [ ]:
# ============================================================
# 1. SETUP  (Colab: ~1-2 min for installs on first run)
# ============================================================
import sys, subprocess, importlib

def _pip(*pkgs):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])

for pkg, mod in [("rdkit", "rdkit"), ("shap", "shap"), ("pubchempy", "pubchempy")]:
    try:
        importlib.import_module(mod)
    except ImportError:
        _pip(pkg)
print("Python", sys.version.split()[0])

In [ ]:
import os, io, re, json, math, time, random, warnings, shutil, platform, itertools
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

from rdkit import Chem, DataStructs, RDLogger
from rdkit.Chem import Descriptors, AllChem, MACCSkeys, Draw, rdFingerprintGenerator
from rdkit.ML.Cluster import Butina

from sklearn.base import clone, BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.compose import TransformedTargetRegressor
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import VarianceThreshold, f_regression
from sklearn.decomposition import PCA
from sklearn.model_selection import KFold, GroupKFold, GridSearchCV
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error, roc_auc_score
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import Ridge
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, GradientBoostingRegressor
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, WhiteKernel, ConstantKernel as CK
import joblib

warnings.filterwarnings("ignore")
RDLogger.DisableLog("rdApp.*")
SEED = 42
random.seed(SEED); np.random.seed(SEED)

# ------------------------------ CONFIG (edit here) ------------------------------
CFG = dict(
    FAST=False,                      # False: paper-grade (final run). True: quick demo (5 CV repeats, 50 y-scrambles)
    USE_UPLOADED_CSV=False,          # True -> you will be asked to upload your curated CSV (Section 2)
    MIN_LABELS=12,                   # min labelled solvents needed to train a QSPR model for a property
    FEATURE_SETS=["RDKit2D", "RDKit2D+MACCS", "Morgan"],
    TARGETS=["bp_C", "flash_C", "log_eps", "dD", "dP", "dH", "alpha", "beta", "pi_star"],
    REFS=["DMF", "NMP", "DMAc"],     # hazardous solvents to be replaced
    BP_WINDOW=(150, 260),            # desired boiling window (deg C) for a low-volatility dipolar aprotic replacement
    FLASH_MIN=60,                    # desired minimum flash point (deg C)
    UNKNOWN_HAZARD=0.5,              # hazard penalty when CMR status is unknown (data-gap penalty)
    SIM_MIN=0.25,                    # min Tanimoto (Morgan) to training set for 'in domain'
    Q2_USABLE=0.5,                   # property models below this CV Q2 are flagged as weak
    WEIGHTS=dict(perf=0.45, hazard=0.25, bio=0.20, window=0.10),   # multi-objective weights
    N_MC=1000,                       # Monte-Carlo draws for rank robustness
)
N_REPEATS = 2 if CFG["FAST"] else 5
N_PERM = 10 if CFG["FAST"] else 50
N_TREES = 150 if CFG["FAST"] else 500

# ------------------------------ output + plotting helpers ------------------------------
OUT = Path("outputs"); (OUT / "figures").mkdir(parents=True, exist_ok=True)
sns.set_theme(context="notebook", style="whitegrid")
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 300, "axes.spines.top": False,
                     "axes.spines.right": False, "font.size": 10})
ROLE_COLORS = {"reference": "#D55E00", "candidate": "#009E73", "conventional": "#7F7F7F"}   # colour-blind safe

def savefig(name):
    for ext in ("png", "pdf"):
        plt.savefig(OUT / "figures" / f"{name}.{ext}", bbox_inches="tight")
print("Config OK | repeats:", N_REPEATS, "| y-scrambles:", N_PERM)

## 2. Data: solvent library and curation

Columns: `name, smiles, role, bio_origin, bp_C, flash_C, eps, dD, dP, dH, alpha, beta, pi_star, cmr, source`

* `role`: `reference` (DMF/NMP/DMAc), `candidate` (bio-based replacements to rank), `conventional` (extra solvents that help the QSPR learn chemistry and act as benchmarks).
* `bio_origin`: `full` (made from biomass today), `potential` (bio route exists), `fossil`.
* `eps` = relative permittivity; `dD/dP/dH` = Hansen parameters (MPa^0.5); `alpha/beta/pi_star` = Kamlet-Taft.
* `cmr`: 1 = classified carcinogenic/mutagenic/reprotoxic, 0 = known not CMR, blank = **unknown** (penalised, not ignored).

**To use your own data:** set `USE_UPLOADED_CSV=True` in the config and re-run; the CSV needs the same columns. Blank cells are fine (each property model uses only the solvents that have that label). Candidates may have *no* labels at all; that is exactly what the QSPR models are for.

In [ ]:
COLS = ["name", "smiles", "role", "bio_origin", "bp_C", "flash_C", "eps", "dD", "dP", "dH",
        "alpha", "beta", "pi_star", "cmr", "source"]

# ---- Curated library (102 solvents). Property values come from the Stenutz "Tables for Chemistry" pages
# ---- (Kamlet-Taft, Hansen, dielectric constant, boiling point); blank = no value retrieved (predicted by QSPR).
# ---- Hansen values converted from (cal/mL)^0.5 to MPa^0.5 (x2.0455). flash_C is intentionally empty.
# ---- cmr verified (supplier SDS / NITE GHS / ECHA) only for rows whose source says "cmr VERIFIED"; all other cmr and all bio_origin flags are unverified.
DATA_CSV = r'''
name,smiles,role,bio_origin,bp_C,flash_C,eps,dD,dP,dH,alpha,beta,pi_star,cmr,source
DMF,CN(C)C=O,reference,fossil,153.0,,36.71,17.4,13.7,11.3,0.0,0.69,0.88,1.0,KT: Stenutz kamlettaft.php; Hansen: Stenutz hansen.php [(cal/mL)^0.5 x2.0455]; eps: Stenutz solv23.php; bp: Stenutz; cmr/bio_origin: not verified here (check ECHA C&L / producer data); cmr 1 SCREENED 2026-11-22: PubChem GHS aggregate H-codes [H226 H303 H312 H313 H315 H318 H319 H320 H331 H332 H341 H350 H360 H360D H370 H371 H372] (CMR code listed); NOT a harmonised ECHA check
NMP,CN1CCCC1=O,reference,fossil,,,32.2,18.0,12.3,7.2,0.0,0.77,0.92,1.0,KT: Stenutz kamlettaft.php; Hansen: Stenutz hansen.php [(cal/mL)^0.5 x2.0455]; eps: Stenutz solv23.php; cmr VERIFIED 2026-11: ECHA C&L notified: Repr. 1B H360D; bio_origin: not verified
DMAc,CC(=O)N(C)C,reference,fossil,166.0,,37.78,16.8,11.5,10.2,0.0,0.76,0.88,1.0,KT: Stenutz kamlettaft.php; Hansen: Stenutz hansen.php [(cal/mL)^0.5 x2.0455]; eps: Stenutz solv23.php; bp: Stenutz; cmr/bio_origin: not verified here (check ECHA C&L / producer data); cmr 1 SCREENED 2026-11-22: PubChem GHS aggregate H-codes [H227 H303 H312 H313 H316 H319 H320 H331 H332 H336 H350 H351 H360 H360D H370 H372 H373] (CMR code listed); NOT a harmonised ECHA check
DMSO,CS(C)=O,conventional,fossil,189.0,,46.68,18.4,16.4,10.2,0.0,0.76,1.0,0.0,KT: Stenutz kamlettaft.php; Hansen: Stenutz hansen.php [(cal/mL)^0.5 x2.0455]; eps: Stenutz solv23.php; bp: Stenutz; cmr/bio_origin: not verified here (check ECHA C&L / producer data)
Acetonitrile,CC#N,conventional,fossil,,,38.8,15.3,18.0,6.1,0.19,0.4,0.75,0.0,KT: Stenutz kamlettaft.php; Hansen: Stenutz hansen.php [(cal/mL)^0.5 x2.0455]; eps: Stenutz solv23.php; cmr/bio_origin: not verified here (check ECHA C&L / producer data)
Acetone,CC(C)=O,conventional,potential,57.0,,21.4,15.5,10.4,7.0,0.08,0.43,0.71,0.0,KT: Stenutz kamlettaft.php; Hansen: Stenutz hansen.php [(cal/mL)^0.5 x2.0455]; eps: Stenutz solv23.php; bp: Stenutz; cmr/bio_origin: not verified here (check ECHA C&L / producer data)
Sulfolane,O=S1(=O)CCCC1,conventional,fossil,285.0,,44.0,18.0,18.0,9.8,0.0,0.39,0.98,,KT: Stenutz kamlettaft.php; Hansen: Stenutz hansen.php [(cal/mL)^0.5 x2.0455]; eps: Stenutz solv23.php; bp: Stenutz; cmr/bio_origin: not verified here (check ECHA C&L / producer data)
THF,C1CCOC1,conventional,potential,,,7.58,16.8,5.7,8.0,0.0,0.55,0.58,1.0,KT: Stenutz kamlettaft.php; Hansen: Stenutz hansen.php [(cal/mL)^0.5 x2.0455]; eps: Stenutz solv23.php; cmr/bio_origin: not verified here (check ECHA C&L / producer data)
Ethyl acetate,CCOC(C)=O,conventional,potential,77.0,,6.4,15.8,5.3,7.2,0.0,0.45,0.55,0.0,KT: Stenutz kamlettaft.php; Hansen: Stenutz hansen.php [(cal/mL)^0.5 x2.0455]; eps: Stenutz solv23.php; bp: Stenutz; cmr/bio_origin: not verified here (check ECHA C&L / producer data)
Dimethyl carbonate,COC(=O)OC,conventional,potential,90.0,,3.17,17.4,9.6,3.9,0.0,0.43,,0.0,KT: Stenutz kamlettaft.php; Hansen: Stenutz hansen.php [(cal/mL)^0.5 x2.0455]; eps: Stenutz solv23.php; bp: Stenutz; cmr/bio_origin: not verified here (check ECHA C&L / producer data)
Toluene,Cc1ccccc1,conventional,fossil,,,2.38,18.0,1.4,2.0,0.0,0.11,0.54,1.0,KT: Stenutz kamlettaft.php; Hansen: Stenutz hansen.php [(cal/mL)^0.5 x2.0455]; eps: Stenutz solv23.php; cmr/bio_origin: not verified here (check ECHA C&L / producer data)
Ethanol,CCO,conventional,full,,,24.5,15.8,8.8,19.4,0.86,0.75,0.54,0.0,KT: Stenutz kamlettaft.php; Hansen: Stenutz hansen.php [(cal/mL)^0.5 x2.0455]; eps: Stenutz solv23.php; cmr/bio_origin: not verified here (check ECHA C&L / producer data)
Methanol,CO,conventional,potential,,,32.6,15.1,12.3,22.3,0.98,0.66,0.6,0.0,KT: Stenutz kamlettaft.php; Hansen: Stenutz hansen.php [(cal/mL)^0.5 x2.0455]; eps: Stenutz solv23.php; cmr/bio_origin: not verified here (check ECHA C&L / producer data)
Dichloromethane,ClCCl,conventional,fossil,,,8.93,18.2,6.3,6.1,0.13,0.1,0.82,1.0,KT: Stenutz kamlettaft.php; Hansen: Stenutz hansen.php [(cal/mL)^0.5 x2.0455]; eps: Stenutz solv23.php; cmr/bio_origin: not verified here (check ECHA C&L / producer data)
Dioxane,C1COCCO1,conventional,fossil,,,2.25,19.0,1.8,7.4,0.0,0.37,0.55,1.0,KT: Stenutz kamlettaft.php; Hansen: Stenutz hansen.php [(cal/mL)^0.5 x2.0455]; eps: Stenutz solv23.php; cmr/bio_origin: not verified here (check ECHA C&L / producer data)
Anisole,COc1ccccc1,conventional,potential,,,4.45,,,,0.0,0.32,0.73,0.0,KT: Stenutz kamlettaft.php; eps: Stenutz solv23.php; cmr/bio_origin: not verified here (check ECHA C&L / producer data)
Cyclohexanone,O=C1CCCCC1,conventional,fossil,,,18.2,17.8,6.3,5.1,0.0,0.53,0.76,0.0,KT: Stenutz kamlettaft.php; Hansen: Stenutz hansen.php [(cal/mL)^0.5 x2.0455]; eps: Stenutz solv23.php; cmr/bio_origin: not verified here (check ECHA C&L / producer data)
2-Butanone,CCC(C)=O,conventional,potential,80.0,,18.5,16.0,9.0,5.1,0.06,0.48,0.67,0.0,KT: Stenutz kamlettaft.php; Hansen: Stenutz hansen.php [(cal/mL)^0.5 x2.0455]; eps: Stenutz solv23.php; bp: Stenutz; cmr/bio_origin: not verified here (check ECHA C&L / producer data)
Pyridine,c1ccncc1,conventional,fossil,,,12.5,19.0,8.8,5.9,0.0,0.64,0.87,,KT: Stenutz kamlettaft.php; Hansen: Stenutz hansen.php [(cal/mL)^0.5 x2.0455]; eps: Stenutz solv23.php; cmr/bio_origin: not verified here (check ECHA C&L / producer data)
gamma-Butyrolactone,O=C1CCCO1,conventional,potential,,,40.96,19.0,16.6,7.4,0.0,0.49,0.87,,KT: Stenutz kamlettaft.php; Hansen: Stenutz hansen.php [(cal/mL)^0.5 x2.0455]; eps: Stenutz solv23.php; cmr/bio_origin: not verified here (check ECHA C&L / producer data)
Cyrene,C1CC(=O)C2COC1O2,candidate,full,,,,,,,,,,0.0,no sourced property values (to be predicted by QSPR); cmr VERIFIED 2026-11: Sigma-Aldrich SDS 807796 (H319; no CMR class); bio_origin: not verified
GVL,CC1CCC(=O)O1,candidate,full,207.0,,36.14,,,,,,,0.0,"eps: Stenutz solv23.php; bp: Stenutz; cmr VERIFIED 2026-11: Sigma-Aldrich SDS 918660 (H315,H319; no CMR class); bio_origin: not verified"
Propylene carbonate,CC1COC(=O)O1,candidate,potential,,,64.9,20.0,18.0,4.1,0.0,0.4,0.83,0.0,"KT: Stenutz kamlettaft.php; Hansen: Stenutz hansen.php [(cal/mL)^0.5 x2.0455]; eps: Stenutz solv23.php; cmr VERIFIED 2026-11: NITE GHS 108-32-7 (repro/carc/mutagenicity: not classifiable, no data; H319); bio_origin: not verified"
2-MeTHF,CC1CCCO1,candidate,full,79.0,,6.97,,,,0.0,0.45,,0.0,"KT: Stenutz kamlettaft.php; eps: Stenutz solv23.php; bp: Stenutz; cmr VERIFIED 2026-11: Sigma-Aldrich SDS 85488 (H225,H302,H315,H318; no CMR class); bio_origin: not verified"
Ethyl lactate,CC(O)C(=O)OCC,candidate,full,154.0,,13.1,16.0,7.6,12.5,,,,0.0,"Hansen: Stenutz hansen.php [(cal/mL)^0.5 x2.0455]; eps: Stenutz solv23.php; bp: Stenutz; cmr VERIFIED 2026-11: Sigma-Aldrich SDS w244007 (H226,H315,H318,H335; no CMR class); bio_origin: not verified"
Dimethyl isosorbide,COC1COC2C1OCC2OC,candidate,full,,,,,,,,,,0.0,no sourced property values (to be predicted by QSPR); cmr 0 VERIFIED 2026-11-25: ECHA C&L Inventory notified entries for EC 226-159-8 / CAS 5306-85-4 (Eye Irrit. 2 H319 and 'not classified'; no harmonised classification) and Spectrum Chemical SDS I2520 (not classified; reproductive toxicity 'no data'); bio_origin: not verified
Glycerol carbonate,OCC1COC(=O)O1,candidate,full,,,,18.0,25.6,17.4,,,,,Hansen: Stenutz hansen.php [(cal/mL)^0.5 x2.0455]; cmr/bio_origin: not verified here (check ECHA C&L / producer data)
TMO,CC1(C)CCC(C)(C)O1,candidate,potential,112.0,,5.03,,,,,,,,eps: Stenutz solv23.php; bp: Stenutz; cmr/bio_origin: not verified here (check ECHA C&L / producer data)
CPME,COC1CCCC1,candidate,potential,,,,,,,,,,0.0,"no sourced property values (to be predicted by QSPR); cmr VERIFIED 2026-11: Sigma-Aldrich SDS 675970 (H225,H302,H315,H319,H402; no CMR class); bio_origin: not verified"
NBP,CCCCN1CCCC1=O,candidate,potential,,,,,,,,,,0.0,"no sourced property values (to be predicted by QSPR); cmr/bio_origin: not verified here (check ECHA C&L / producer data); NBP cmr left blank: TCI page lists H315,H319 only but SDS unavailable; cmr 0 SCREENED 2026-11-22: PubChem GHS aggregate H-codes [H315 H319 H335] (H-codes listed, none CMR); NOT a harmonised ECHA check"
PolarClean,COC(=O)C(C)CCC(=O)N(C)C,candidate,fossil,,,,,,,,,,,no sourced property values (to be predicted by QSPR); cmr/bio_origin: not verified here (check ECHA C&L / producer data)
Methyl levulinate,COC(=O)CCC(C)=O,candidate,full,,,,,,,,,,0.0,no sourced property values (to be predicted by QSPR); cmr VERIFIED 2026-11: TCI product page L0044 (H227 only; SDS not on page); bio_origin: not verified
Ethyl levulinate,CCOC(=O)CCC(C)=O,candidate,full,206.0,,11.9,,,,,,,,eps: Stenutz solv23.php; bp: Stenutz; cmr/bio_origin: not verified here (check ECHA C&L / producer data)
Furfural,O=Cc1ccco1,candidate,full,161.0,,42.0,,,,,,,1.0,eps: Stenutz solv23.php; bp: Stenutz; cmr/bio_origin: not verified here (check ECHA C&L / producer data); cmr 1 SCREENED 2026-11-22: PubChem GHS aggregate H-codes [H226 H301 H311 H312 H315 H319 H330 H331 H335 H351 H370 H372 H373 H402 H412] (CMR code listed); NOT a harmonised ECHA check
Dimethyl succinate,COC(=O)CCC(=O)OC,candidate,potential,195.0,,5.1,,,,,,,0.0,"eps: Stenutz solv23.php; bp: Stenutz; cmr VERIFIED 2026-11: TCI product page S0104 (H319,H373; no CMR H-code); bio_origin: not verified"
Diethyl carbonate,CCOC(=O)OCC,candidate,potential,127.0,,3.1,16.6,3.1,6.1,0.0,0.4,0.45,0.0,"KT: Stenutz kamlettaft.php; Hansen: Stenutz hansen.php [(cal/mL)^0.5 x2.0455]; eps: Stenutz solv23.php; bp: Stenutz; cmr/bio_origin: not verified here (check ECHA C&L / producer data); cmr 0 SCREENED 2026-11-22: PubChem GHS aggregate H-codes [H226 H315 H319 H335] (H-codes listed, none CMR); NOT a harmonised ECHA check"
Triacetin,CC(=O)OCC(COC(C)=O)OC(C)=O,candidate,full,259.0,,7.01,,,,,,,,eps: Stenutz solv23.php; bp: Stenutz; cmr/bio_origin: not verified here (check ECHA C&L / producer data)
Butyl lactate,CCCCOC(=O)C(C)O,candidate,full,,,,15.6,6.5,10.2,,,,0.0,"Hansen: Stenutz hansen.php [(cal/mL)^0.5 x2.0455]; cmr/bio_origin: not verified here (check ECHA C&L / producer data); cmr 0 SCREENED 2026-11-22: PubChem GHS aggregate H-codes [H227 H315 H318 H319 H335 H336] (H-codes listed, none CMR); NOT a harmonised ECHA check"
Tetrahydrofurfuryl alcohol,OCC1CCCO1,candidate,full,,,13.48,17.8,8.2,12.9,,,,1.0,Hansen: Stenutz hansen.php [(cal/mL)^0.5 x2.0455]; eps: Stenutz solv23.php; cmr/bio_origin: not verified here (check ECHA C&L / producer data); cmr 1 SCREENED 2026-11-22: PubChem GHS aggregate H-codes [H227 H315 H319 H336 H360 H360Df H361 H373] (CMR code listed); NOT a harmonised ECHA check
Dimethyllactamide,CC(O)C(=O)N(C)C,candidate,full,,,,,,,,,,,no sourced property values (to be predicted by QSPR); cmr/bio_origin: not verified here (check ECHA C&L / producer data)
Limonene,CC1=CCC(CC1)C(C)=C,candidate,full,175.0,,2.36,17.2,1.8,4.3,,,,0.0,"Hansen: Stenutz hansen.php [(cal/mL)^0.5 x2.0455]; eps: Stenutz solv23.php; bp: Stenutz; cmr/bio_origin: not verified here (check ECHA C&L / producer data); cmr 0 SCREENED 2026-11-22: PubChem GHS aggregate H-codes [H226 H304 H315 H317 H319 H400 H410] (H-codes listed, none CMR); NOT a harmonised ECHA check"
p-Cymene,Cc1ccc(C(C)C)cc1,candidate,full,177.0,,2.24,,,,,,,0.0,"eps: Stenutz solv23.php; bp: Stenutz; cmr/bio_origin: not verified here (check ECHA C&L / producer data); cmr 0 SCREENED 2026-11-22: PubChem GHS aggregate H-codes [H226 H304 H315 H319 H331 H336 H401 H411 H412] (H-codes listed, none CMR); NOT a harmonised ECHA check"
Dioxolane,C1COCO1,candidate,potential,,,,18.0,6.5,9.2,0.0,0.45,0.69,1.0,KT: Stenutz kamlettaft.php; Hansen: Stenutz hansen.php [(cal/mL)^0.5 x2.0455]; cmr/bio_origin: not verified here (check ECHA C&L / producer data); cmr 1 SCREENED 2026-11-22: PubChem GHS aggregate H-codes [H225 H303 H316 H318 H319 H341 H360 H361] (CMR code listed); NOT a harmonised ECHA check
Formamide,NC=O,conventional,fossil,,,111.0,17.2,26.2,19.0,0.71,0.48,0.97,1.0,KT: Stenutz kamlettaft.php; Hansen: Stenutz hansen.php [(cal/mL)^0.5 x2.0455]; eps: Stenutz solv23.php; cmr/bio_origin: not verified here (check ECHA C&L / producer data)
N-Methylformamide,CNC=O,conventional,fossil,,,171.0,,,,0.62,0.8,0.9,1.0,KT: Stenutz kamlettaft.php; eps: Stenutz solv23.php; cmr/bio_origin: not verified here (check ECHA C&L / producer data)
"N,N-Dimethylpropionamide",CCC(=O)N(C)C,conventional,fossil,,,33.08,,,,,,,,eps: Stenutz solv23.php; cmr/bio_origin: not verified here (check ECHA C&L / producer data)
Tetramethylurea,CN(C)C(=O)N(C)C,conventional,fossil,177.0,,24.46,,,,0.0,0.8,0.83,1.0,KT: Stenutz kamlettaft.php; eps: Stenutz solv23.php; bp: Stenutz; cmr/bio_origin: not verified here (check ECHA C&L / producer data)
DMPU,CN1CCCN(C)C1=O,conventional,fossil,246.0,,,,,,,,,,bp: Stenutz; cmr/bio_origin: not verified here (check ECHA C&L / producer data)
"DMI (1,3-dimethylimidazolidinone)",CN1CCN(C)C1=O,conventional,fossil,,,,,,,,,,,no sourced property values (to be predicted by QSPR); cmr/bio_origin: not verified here (check ECHA C&L / producer data)
HMPA,CN(C)P(=O)(N(C)C)N(C)C,conventional,fossil,,,30.0,,,,0.0,1.05,0.87,1.0,KT: Stenutz kamlettaft.php; eps: Stenutz solv23.php; cmr/bio_origin: not verified here (check ECHA C&L / producer data)
2-Pyrrolidone,O=C1CCCN1,conventional,fossil,,,,,,,0.36,0.77,0.85,1.0,KT: Stenutz kamlettaft.php; cmr VERIFIED 2026-11: Sigma-Aldrich SDS 83300 (H360 Repr. 1B); bio_origin: not verified
NEP,CCN1CCCC1=O,conventional,fossil,,,,,,,,,,1.0,no sourced property values (to be predicted by QSPR); cmr/bio_origin: not verified here (check ECHA C&L / producer data)
NOP,CCCCCCCCN1CCCC1=O,candidate,potential,,,,,,,,,,,no sourced property values (to be predicted by QSPR); cmr/bio_origin: not verified here (check ECHA C&L / producer data)
N-Formylmorpholine,O=CN1CCOCC1,conventional,fossil,,,,,,,,,,,no sourced property values (to be predicted by QSPR); cmr/bio_origin: not verified here (check ECHA C&L / producer data)
Nitromethane,C[N+](=O)[O-],conventional,fossil,,,39.4,15.8,18.8,5.1,0.22,0.06,0.85,,KT: Stenutz kamlettaft.php; Hansen: Stenutz hansen.php [(cal/mL)^0.5 x2.0455]; eps: Stenutz solv23.php; cmr/bio_origin: not verified here (check ECHA C&L / producer data)
Nitrobenzene,[O-][N+](=O)c1ccccc1,conventional,fossil,,,34.8,20.0,8.6,4.1,0.0,0.3,1.01,1.0,KT: Stenutz kamlettaft.php; Hansen: Stenutz hansen.php [(cal/mL)^0.5 x2.0455]; eps: Stenutz solv23.php; cmr/bio_origin: not verified here (check ECHA C&L / producer data)
Propionitrile,CCC#N,conventional,fossil,,,27.7,,,,0.0,0.39,0.71,,KT: Stenutz kamlettaft.php; eps: Stenutz solv23.php; cmr/bio_origin: not verified here (check ECHA C&L / producer data)
Benzonitrile,N#Cc1ccccc1,conventional,fossil,190.0,,26.0,,,,0.0,0.37,0.9,,KT: Stenutz kamlettaft.php; eps: Stenutz solv23.php; bp: Stenutz; cmr/bio_origin: not verified here (check ECHA C&L / producer data)
Ethylene carbonate,O=C1OCCO1,conventional,potential,261.0,,,19.4,21.7,5.1,0.0,0.41,,,KT: Stenutz kamlettaft.php; Hansen: Stenutz hansen.php [(cal/mL)^0.5 x2.0455]; bp: Stenutz; cmr/bio_origin: not verified here (check ECHA C&L / producer data)
Acetophenone,CC(=O)c1ccccc1,conventional,fossil,,,17.4,19.6,8.6,3.7,0.04,0.49,0.9,,KT: Stenutz kamlettaft.php; Hansen: Stenutz hansen.php [(cal/mL)^0.5 x2.0455]; eps: Stenutz solv23.php; cmr/bio_origin: not verified here (check ECHA C&L / producer data)
Cyclopentanone,O=C1CCCC1,conventional,potential,,,14.45,,,,0.0,0.52,0.76,,KT: Stenutz kamlettaft.php; eps: Stenutz solv23.php; cmr/bio_origin: not verified here (check ECHA C&L / producer data)
MIBK,CC(C)CC(C)=O,conventional,fossil,,,13.11,15.3,6.1,4.1,0.02,0.48,0.65,,KT: Stenutz kamlettaft.php; Hansen: Stenutz hansen.php [(cal/mL)^0.5 x2.0455]; eps: Stenutz solv23.php; cmr/bio_origin: not verified here (check ECHA C&L / producer data)
Diacetone alcohol,CC(=O)CC(C)(C)O,conventional,fossil,,,18.3,15.8,8.2,10.8,,,,,Hansen: Stenutz hansen.php [(cal/mL)^0.5 x2.0455]; eps: Stenutz solv23.php; cmr/bio_origin: not verified here (check ECHA C&L / producer data)
Diethyl ether,CCOCC,conventional,potential,35.0,,4.33,14.5,2.9,5.1,0.0,0.47,0.27,,KT: Stenutz kamlettaft.php; Hansen: Stenutz hansen.php [(cal/mL)^0.5 x2.0455]; eps: Stenutz solv23.php; bp: Stenutz; cmr/bio_origin: not verified here (check ECHA C&L / producer data)
MTBE,COC(C)(C)C,conventional,fossil,,,4.5,,,,,,,,eps: Stenutz solv23.php; cmr/bio_origin: not verified here (check ECHA C&L / producer data)
"1,2-Dimethoxyethane",COCCOC,conventional,fossil,,,,,,,0.0,0.41,0.53,1.0,KT: Stenutz kamlettaft.php; cmr/bio_origin: not verified here (check ECHA C&L / producer data)
Diglyme,COCCOCCOC,conventional,fossil,,,7.3,,,,0.0,0.4,0.64,1.0,KT: Stenutz kamlettaft.php; eps: Stenutz solv23.php; cmr/bio_origin: not verified here (check ECHA C&L / producer data)
Methyl acetate,COC(C)=O,conventional,potential,57.0,,6.7,15.5,7.2,7.6,0.0,0.42,0.6,,KT: Stenutz kamlettaft.php; Hansen: Stenutz hansen.php [(cal/mL)^0.5 x2.0455]; eps: Stenutz solv23.php; bp: Stenutz; cmr/bio_origin: not verified here (check ECHA C&L / producer data)
Butyl acetate,CCCCOC(C)=O,conventional,potential,126.0,,5.1,15.8,3.7,6.3,0.0,0.45,0.46,,KT: Stenutz kamlettaft.php; Hansen: Stenutz hansen.php [(cal/mL)^0.5 x2.0455]; eps: Stenutz solv23.php; bp: Stenutz; cmr/bio_origin: not verified here (check ECHA C&L / producer data)
Isopropyl acetate,CC(C)OC(C)=O,conventional,potential,,,,14.9,4.5,8.2,,,,,Hansen: Stenutz hansen.php [(cal/mL)^0.5 x2.0455]; cmr/bio_origin: not verified here (check ECHA C&L / producer data)
Hexane,CCCCCC,conventional,fossil,,,1.9,14.9,0.0,0.0,0.0,0.0,-0.04,1.0,KT: Stenutz kamlettaft.php; Hansen: Stenutz hansen.php [(cal/mL)^0.5 x2.0455]; eps: Stenutz solv23.php; cmr/bio_origin: not verified here (check ECHA C&L / producer data)
Heptane,CCCCCCC,conventional,fossil,,,1.92,15.3,0.0,0.0,0.0,0.0,-0.08,,KT: Stenutz kamlettaft.php; Hansen: Stenutz hansen.php [(cal/mL)^0.5 x2.0455]; eps: Stenutz solv23.php; cmr/bio_origin: not verified here (check ECHA C&L / producer data)
Cyclohexane,C1CCCCC1,conventional,fossil,,,1.88,16.8,0.0,0.2,0.0,0.0,0.0,,KT: Stenutz kamlettaft.php; Hansen: Stenutz hansen.php [(cal/mL)^0.5 x2.0455]; eps: Stenutz solv23.php; cmr/bio_origin: not verified here (check ECHA C&L / producer data)
Benzene,c1ccccc1,conventional,fossil,,,2.28,18.4,0.0,2.0,0.0,0.1,0.59,1.0,KT: Stenutz kamlettaft.php; Hansen: Stenutz hansen.php [(cal/mL)^0.5 x2.0455]; eps: Stenutz solv23.php; cmr/bio_origin: not verified here (check ECHA C&L / producer data)
p-Xylene,Cc1ccc(C)cc1,conventional,fossil,,,,17.6,1.0,3.1,,,,,Hansen: Stenutz hansen.php [(cal/mL)^0.5 x2.0455]; cmr/bio_origin: not verified here (check ECHA C&L / producer data)
Chloroform,ClC(Cl)Cl,conventional,fossil,,,4.81,17.8,3.1,5.7,0.2,0.1,0.58,1.0,KT: Stenutz kamlettaft.php; Hansen: Stenutz hansen.php [(cal/mL)^0.5 x2.0455]; eps: Stenutz solv23.php; cmr/bio_origin: not verified here (check ECHA C&L / producer data)
"1,2-Dichloroethane",ClCCCl,conventional,fossil,,,,19.0,7.4,4.1,0.0,0.1,0.81,1.0,KT: Stenutz kamlettaft.php; Hansen: Stenutz hansen.php [(cal/mL)^0.5 x2.0455]; cmr/bio_origin: not verified here (check ECHA C&L / producer data)
Chlorobenzene,Clc1ccccc1,conventional,fossil,,,5.62,19.0,4.3,2.0,0.0,0.07,0.71,,KT: Stenutz kamlettaft.php; Hansen: Stenutz hansen.php [(cal/mL)^0.5 x2.0455]; eps: Stenutz solv23.php; cmr/bio_origin: not verified here (check ECHA C&L / producer data)
Isopropanol,CC(C)O,conventional,potential,,,19.92,15.8,6.1,16.4,0.76,0.84,0.48,,KT: Stenutz kamlettaft.php; Hansen: Stenutz hansen.php [(cal/mL)^0.5 x2.0455]; eps: Stenutz solv23.php; cmr/bio_origin: not verified here (check ECHA C&L / producer data)
1-Propanol,CCCO,conventional,potential,,,21.8,16.0,6.8,17.4,0.84,0.9,0.52,,KT: Stenutz kamlettaft.php; Hansen: Stenutz hansen.php [(cal/mL)^0.5 x2.0455]; eps: Stenutz solv23.php; cmr/bio_origin: not verified here (check ECHA C&L / producer data)
1-Butanol,CCCCO,conventional,potential,,,17.8,16.0,5.7,15.8,0.84,0.84,0.47,,KT: Stenutz kamlettaft.php; Hansen: Stenutz hansen.php [(cal/mL)^0.5 x2.0455]; eps: Stenutz solv23.php; cmr/bio_origin: not verified here (check ECHA C&L / producer data)
tert-Butanol,CC(C)(C)O,conventional,potential,,,10.9,15.1,5.1,14.7,0.42,0.93,0.41,,KT: Stenutz kamlettaft.php; Hansen: Stenutz hansen.php [(cal/mL)^0.5 x2.0455]; eps: Stenutz solv23.php; cmr/bio_origin: not verified here (check ECHA C&L / producer data)
Ethylene glycol,OCCO,conventional,potential,,,37.0,17.0,11.0,26.0,0.9,0.52,0.92,,KT: Stenutz kamlettaft.php; Hansen: Stenutz hansen.php [(cal/mL)^0.5 x2.0455]; eps: Stenutz solv23.php; cmr/bio_origin: not verified here (check ECHA C&L / producer data)
Propylene glycol,CC(O)CO,conventional,full,,,,16.8,9.4,23.3,,,,,Hansen: Stenutz hansen.php [(cal/mL)^0.5 x2.0455]; cmr/bio_origin: not verified here (check ECHA C&L / producer data)
"1,3-Propanediol",OCCCO,candidate,full,,,,,,,,,,0.0,"no sourced property values (to be predicted by QSPR); cmr/bio_origin: not verified here (check ECHA C&L / producer data); cmr 0 SCREENED 2026-11-22: PubChem GHS aggregate H-codes [H315] (H-codes listed, none CMR); NOT a harmonised ECHA check"
Glycerol,OCC(O)CO,candidate,full,,,,17.4,12.1,29.3,,,,,Hansen: Stenutz hansen.php [(cal/mL)^0.5 x2.0455]; cmr/bio_origin: not verified here (check ECHA C&L / producer data)
2-Methoxyethanol,COCCO,conventional,fossil,,,16.93,16.2,9.2,16.4,,,,1.0,Hansen: Stenutz hansen.php [(cal/mL)^0.5 x2.0455]; eps: Stenutz solv23.php; cmr/bio_origin: not verified here (check ECHA C&L / producer data)
Diethylene glycol,OCCOCCO,conventional,fossil,,,31.7,16.2,14.7,20.5,,,,,Hansen: Stenutz hansen.php [(cal/mL)^0.5 x2.0455]; eps: Stenutz solv23.php; cmr/bio_origin: not verified here (check ECHA C&L / producer data)
Water,O,conventional,full,,,80.1,15.6,16.0,42.3,1.17,0.47,1.09,0.0,KT: Stenutz kamlettaft.php; Hansen: Stenutz hansen.php [(cal/mL)^0.5 x2.0455]; eps: Stenutz solv23.php; cmr/bio_origin: not verified here (check ECHA C&L / producer data)
Acetic acid,CC(=O)O,conventional,potential,,,6.2,14.5,8.0,13.5,1.12,0.45,0.64,,KT: Stenutz kamlettaft.php; Hansen: Stenutz hansen.php [(cal/mL)^0.5 x2.0455]; eps: Stenutz solv23.php; cmr/bio_origin: not verified here (check ECHA C&L / producer data)
Triethylamine,CCN(CC)CC,conventional,fossil,,,3.2,,,,0.0,0.71,0.14,,KT: Stenutz kamlettaft.php; eps: Stenutz solv23.php; cmr/bio_origin: not verified here (check ECHA C&L / producer data)
Methyl lactate,COC(=O)C(C)O,candidate,full,,,,,,,,,,0.0,"no sourced property values (to be predicted by QSPR); cmr/bio_origin: not verified here (check ECHA C&L / producer data); cmr 0 SCREENED 2026-11-22: PubChem GHS aggregate H-codes [H226 H319 H335] (H-codes listed, none CMR); NOT a harmonised ECHA check"
Levulinic acid,CC(=O)CCC(=O)O,candidate,full,,,,,,,,,,0.0,"no sourced property values (to be predicted by QSPR); cmr/bio_origin: not verified here (check ECHA C&L / producer data); cmr 0 SCREENED 2026-11-22: PubChem GHS aggregate H-codes [H302 H315 H317 H318 H319] (H-codes listed, none CMR); NOT a harmonised ECHA check"
Diethyl succinate,CCOC(=O)CCC(=O)OCC,candidate,full,,,6.64,,,,,,,,eps: Stenutz solv23.php; cmr/bio_origin: not verified here (check ECHA C&L / producer data)
Dimethyl glutarate,COC(=O)CCCC(=O)OC,candidate,potential,,,,,,,,,,0.0,"no sourced property values (to be predicted by QSPR); cmr/bio_origin: not verified here (check ECHA C&L / producer data); cmr 0 SCREENED 2026-11-22: PubChem GHS aggregate H-codes [H336 H373 H402] (H-codes listed, none CMR); NOT a harmonised ECHA check"
Dimethyl adipate,COC(=O)CCCCC(=O)OC,candidate,potential,,,,16.9,4.3,9.2,,,,0.0,"Hansen: Stenutz hansen.php [(cal/mL)^0.5 x2.0455]; cmr/bio_origin: not verified here (check ECHA C&L / producer data); cmr 0 SCREENED 2026-11-22: PubChem GHS aggregate H-codes [H319] (H-codes listed, none CMR); NOT a harmonised ECHA check"
Dimethyl 2-methylglutarate,COC(=O)CCC(C)C(=O)OC,candidate,fossil,,,,,,,,,,,no sourced property values (to be predicted by QSPR); cmr/bio_origin: not verified here (check ECHA C&L / producer data)
Furfuryl alcohol,OCc1ccco1,candidate,full,176.0,,,,,,,,,1.0,bp: Stenutz; cmr/bio_origin: not verified here (check ECHA C&L / producer data); cmr 1 SCREENED 2026-11-22: PubChem GHS aggregate H-codes [H227 H301 H302 H311 H312 H315 H317 H319 H330 H331 H335 H336 H351 H372 H373] (CMR code listed); NOT a harmonised ECHA check
2-Methylfuran,Cc1ccco1,candidate,full,,,,,,,,,,0.0,"no sourced property values (to be predicted by QSPR); cmr/bio_origin: not verified here (check ECHA C&L / producer data); NITE GHS: repro/carc 'classification not possible' (no data) -> cmr left blank; cmr 0 SCREENED 2026-11-22: PubChem GHS aggregate H-codes [H225 H301 H319 H330 H331 H411] (H-codes listed, none CMR); NOT a harmonised ECHA check"
"2,5-Dimethylfuran",Cc1ccc(C)o1,candidate,full,,,,,,,,,,0.0,"no sourced property values (to be predicted by QSPR); cmr/bio_origin: not verified here (check ECHA C&L / producer data); cmr 0 SCREENED 2026-11-22: PubChem GHS aggregate H-codes [H225 H302] (H-codes listed, none CMR); NOT a harmonised ECHA check"
"1,8-Cineole",CC1(C2CCC(O1)(CC2)C)C,candidate,full,,,,,,,,,,0.0,"no sourced property values (to be predicted by QSPR); cmr/bio_origin: not verified here (check ECHA C&L / producer data); cmr 0 SCREENED 2026-11-22: PubChem GHS aggregate H-codes [H226 H317] (H-codes listed, none CMR); NOT a harmonised ECHA check"
alpha-Pinene,CC1=CCC2CC1C2(C)C,candidate,full,,,2.64,,,,,,,0.0,"eps: Stenutz solv23.php; cmr/bio_origin: not verified here (check ECHA C&L / producer data); cmr 0 SCREENED 2026-11-22: PubChem GHS aggregate H-codes [H226 H302 H304 H315 H317 H318 H371 H372 H373 H400 H410] (H-codes listed, none CMR); NOT a harmonised ECHA check"
'''
default_df = pd.read_csv(io.StringIO(DATA_CSV.strip()))

if CFG["USE_UPLOADED_CSV"]:
    from google.colab import files
    up = files.upload()
    raw = pd.read_csv(next(iter(up)))
    if "source" not in raw.columns:
        raw["source"] = "user CSV"
else:
    raw = default_df.copy()

missing_cols = [c for c in COLS if c not in raw.columns]
assert not missing_cols, f"CSV is missing columns: {missing_cols}"
num_cols = ["bp_C", "flash_C", "eps", "dD", "dP", "dH", "alpha", "beta", "pi_star", "cmr"]
raw[num_cols] = raw[num_cols].apply(pd.to_numeric, errors="coerce")
print("Rows loaded:", len(raw), "| source:", "uploaded CSV" if CFG["USE_UPLOADED_CSV"] else "built-in curated table")

In [ ]:
# ---------------- structure curation ----------------
def largest_fragment(mol):
    frags = Chem.GetMolFrags(mol, asMols=True)
    return max(frags, key=lambda m: m.GetNumHeavyAtoms())

keep_idx, mols, bad = [], [], []
for i, r in raw.iterrows():
    m = Chem.MolFromSmiles(str(r["smiles"]))
    if m is None:
        bad.append(r["name"]); continue
    keep_idx.append(i); mols.append(largest_fragment(m))
if bad:
    print("Invalid SMILES dropped:", bad)

df = raw.loc[keep_idx].reset_index(drop=True).copy()
df["smiles_can"] = [Chem.MolToSmiles(m) for m in mols]
df["inchikey"] = [Chem.MolToInchiKey(m) for m in mols]
dup = df.duplicated("inchikey", keep="first").values
if dup.any():
    print("Duplicate structures removed:", df.loc[dup, "name"].tolist())
df = df[~dup].reset_index(drop=True)
mols = [m for m, d in zip(mols, dup) if not d]
assert df["name"].is_unique, "solvent names must be unique"
df["log_eps"] = np.log10(df["eps"])          # permittivity spans 2-80 -> model log10(eps)

# ---------------- label audit ----------------
audit = pd.DataFrame({"n_labeled": [int(df[t].notna().sum()) for t in CFG["TARGETS"]]}, index=CFG["TARGETS"])
audit["trainable"] = audit["n_labeled"] >= CFG["MIN_LABELS"]
print(f"{len(df)} solvents | roles:", df["role"].value_counts().to_dict())
display(audit)
if audit["n_labeled"].max() < 100:
    print("NOTE: <100 labelled solvents per property -> demo scale. Expand the dataset for a publication-grade study.")

In [ ]:
# ---------------- optional: look at the structures ----------------
try:
    sub = df[df.role != "conventional"]
    img = Draw.MolsToGridImage([mols[i] for i in sub.index], legends=sub["name"].tolist(), molsPerRow=6, subImgSize=(190, 150))
    display(img)
except Exception as e:
    print("Structure grid skipped:", e)

# ---------------- helper to grow the library: name -> SMILES via PubChem ----------------
def pubchem_smiles(name, retries=2):
    import pubchempy as pcp
    for _ in range(retries):
        try:
            hits = pcp.get_compounds(name, "name")
            if hits:
                h = hits[0]
                return getattr(h, "connectivity_smiles", None) or getattr(h, "canonical_smiles", None) or h.isomeric_smiles
        except Exception:
            time.sleep(1)
    return None
# Example (needs internet; always check the structure by eye):
# pubchem_smiles("2-methyltetrahydrofuran")

## 3. Featurization

Three feature families are compared (a feature-set ablation is itself a result worth reporting):

* **RDKit2D**: ~200 physchem/topological descriptors (filtered: >10% NaN, constant, |r|>0.95 and extreme-range columns removed. This filtering uses **no labels**, so no leakage).
* **RDKit2D+MACCS**: adds 166 substructure keys.
* **Morgan**: radius-2 count fingerprints (512 bins).
* Optional **QM descriptors** (dipole, polarizability, HOMO/LUMO...) computed with ORCA: put them in `qm_descriptors.csv` (column `name` + numeric columns). Section 10 generates the ORCA inputs. A hybrid physics+ML descriptor set is a strong novelty angle for the paper.

Feature *selection*, scaling and imputation all happen **inside** the CV pipelines, so test folds never influence them.

In [ ]:
def rdkit2d(mol):
    out = {}
    for n, f in Descriptors.descList:
        try:
            out[n] = f(mol)
        except Exception:
            out[n] = np.nan
    return out

D = pd.DataFrame([rdkit2d(m) for m in mols], index=df["name"])
D = D.replace([np.inf, -np.inf], np.nan)
D = D.loc[:, D.isna().mean() <= 0.10]
D = D.loc[:, D.nunique() > 1]
D = D.loc[:, D.abs().max() < 1e6]
corr = D.corr().abs()
upper = corr.where(np.triu(np.ones(corr.shape, dtype=bool), k=1))
D = D.drop(columns=[c for c in upper.columns if (upper[c] > 0.95).any()])

maccs = pd.DataFrame([list(MACCSkeys.GenMACCSKeys(m)) for m in mols], index=df["name"]).add_prefix("maccs_")
maccs = maccs.loc[:, maccs.nunique() > 1]

fp_gen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=512)
morgan = pd.DataFrame(np.array([fp_gen.GetCountFingerprintAsNumPy(m) for m in mols]), index=df["name"]).add_prefix("mc_")
morgan = morgan.loc[:, morgan.nunique() > 1]
fps = [fp_gen.GetFingerprint(m) for m in mols]          # bit vectors for Tanimoto / clustering

FEATURESETS = {"RDKit2D": D, "RDKit2D+MACCS": pd.concat([D, maccs], axis=1), "Morgan": morgan}

# optional QM descriptors from ORCA (see Section 10)
if Path("qm_descriptors.csv").exists():
    Q = pd.read_csv("qm_descriptors.csv").set_index("name").add_prefix("qm_")
    Q = Q.reindex(df["name"])
    if Q.notna().all(axis=1).mean() > 0.5:
        FEATURESETS["RDKit2D+QM"] = pd.concat([D, Q], axis=1)
        CFG["FEATURE_SETS"] = CFG["FEATURE_SETS"] + ["RDKit2D+QM"]
        print("QM descriptors found and added as feature set 'RDKit2D+QM'")

for k, v in FEATURESETS.items():
    print(f"{k:15s} {v.shape[1]:4d} features x {v.shape[0]} solvents")

In [ ]:
# ---------------- Figure 1: chemical space of the library ----------------
Z = StandardScaler().fit_transform(SimpleImputer(strategy="median").fit_transform(FEATURESETS["RDKit2D"].values))
pca = PCA(n_components=2, random_state=SEED).fit(Z)
P2 = pca.transform(Z)
fig, ax = plt.subplots(figsize=(7.5, 5.5))
for role, g in df.groupby("role"):
    ax.scatter(P2[g.index, 0], P2[g.index, 1], s=50, c=ROLE_COLORS.get(role, "k"), label=role,
               alpha=0.9, edgecolor="white", lw=0.6)
for i, n in enumerate(df["name"]):
    ax.annotate(n, (P2[i, 0], P2[i, 1]), fontsize=6, alpha=0.8, xytext=(3, 3), textcoords="offset points")
ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.0f}%)"); ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.0f}%)")
ax.set_title("Chemical space of the solvent library (RDKit descriptors)"); ax.legend(frameon=False)
savefig("fig1_chemical_space"); plt.show()

## 4. QSPR benchmark (nested, repeated cross-validation)

For every property with enough labels we compare a **mean-predictor baseline**, Ridge, SVR, GPR, Random Forest, Extra Trees and Gradient Boosting on each feature set.

* **Outer loop**: repeated 5-fold CV -> out-of-fold (OOF) predictions -> RMSE, MAE, Q2 (= CV R2), Spearman rho.
* **Inner loop**: 3-fold grid search for hyper-parameters (and number of selected descriptors) using the training fold only. This is *nested* CV, so the reported numbers are not inflated by tuning.
* Targets are standardised inside the pipeline (so SVR/GPR hyper-parameters behave sensibly for bp in deg C and alpha in 0-1 alike).

In [ ]:
class KBestSafe(BaseEstimator, TransformerMixin):
    # univariate top-k selector that never asks for more features than exist
    def __init__(self, k=10):
        self.k = k
    def fit(self, X, y):
        F, _ = f_regression(X, y)
        F = np.nan_to_num(F, nan=0.0, posinf=0.0, neginf=0.0)
        self.idx_ = np.argsort(F)[::-1][:min(self.k, X.shape[1])]
        return self
    def transform(self, X):
        return X[:, self.idx_]

def make_zoo():
    def pipe(est, select=True):
        steps = [("imp", SimpleImputer(strategy="median")), ("var", VarianceThreshold(0.0)), ("sc", StandardScaler())]
        if select:
            steps.append(("sel", KBestSafe()))
        steps.append(("est", est))
        return TransformedTargetRegressor(regressor=Pipeline(steps), transformer=StandardScaler())
    g = lambda d: {f"regressor__{k}": v for k, v in d.items()}
    kgrid = [5, 10, 20]
    return {
        "Baseline(mean)": (pipe(DummyRegressor(strategy="mean"), select=False), {}),
        "Ridge": (pipe(Ridge()), g({"sel__k": kgrid, "est__alpha": [0.1, 1, 10, 100]})),
        "SVR-RBF": (pipe(SVR(kernel="rbf", epsilon=0.05)), g({"sel__k": kgrid, "est__C": [1, 10, 100], "est__gamma": ["scale", 0.01, 0.1]})),
        "GPR": (pipe(GaussianProcessRegressor(kernel=CK(1.0) * RBF(5.0) + WhiteKernel(0.1), n_restarts_optimizer=1, random_state=SEED)),
                g({"sel__k": kgrid})),
        "RandomForest": (pipe(RandomForestRegressor(n_estimators=N_TREES, random_state=SEED, n_jobs=-1), select=False), {}),
        "ExtraTrees": (pipe(ExtraTreesRegressor(n_estimators=N_TREES, random_state=SEED, n_jobs=-1), select=False), {}),
        "GradBoost": (pipe(GradientBoostingRegressor(n_estimators=200, max_depth=2, learning_rate=0.05, subsample=0.8, random_state=SEED), select=False), {}),
    }

def nested_eval(X, y, model, grid, repeat_splits, inner_seed=SEED):
    oof = np.zeros((len(repeat_splits), len(y)))
    for r, splits in enumerate(repeat_splits):
        for tr, te in splits:
            est = clone(model)
            if grid:
                inner = KFold(n_splits=min(3, len(tr)), shuffle=True, random_state=inner_seed)
                est = GridSearchCV(est, grid, cv=inner, scoring="neg_root_mean_squared_error", n_jobs=-1)
            est.fit(X[tr], y[tr])
            oof[r, te] = est.predict(X[te])
    return oof

def summarize(y, oof):
    rm = [math.sqrt(mean_squared_error(y, o)) for o in oof]
    q2 = [r2_score(y, o) for o in oof]
    return dict(RMSE=float(np.mean(rm)), RMSE_sd=float(np.std(rm)),
                MAE=float(np.mean([mean_absolute_error(y, o) for o in oof])),
                Q2=float(np.mean(q2)), Q2_sd=float(np.std(q2)),
                rho=float(stats.spearmanr(y, oof.mean(axis=0))[0]))

def kfold_repeats(n, n_repeats, n_splits=5):
    k = min(n_splits, n)
    return [list(KFold(k, shuffle=True, random_state=SEED + r).split(np.arange(n))) for r in range(n_repeats)]

def fit_final(X, y, model, grid):
    if grid:
        gs = GridSearchCV(model, grid, cv=KFold(min(5, len(y)), shuffle=True, random_state=SEED),
                          scoring="neg_root_mean_squared_error", n_jobs=-1).fit(X, y)
        return gs.best_estimator_, gs.best_params_
    return clone(model).fit(X, y), {}

ZOO = make_zoo()
TRAINABLE = [t for t in CFG["TARGETS"] if df[t].notna().sum() >= CFG["MIN_LABELS"]]
print("Trainable targets:", TRAINABLE)

In [ ]:
rows, OOF = [], {}
t0 = time.time()
for t in TRAINABLE:
    m = df[t].notna().values
    y = df.loc[m, t].values.astype(float)
    splits = kfold_repeats(len(y), N_REPEATS)
    for fs in CFG["FEATURE_SETS"]:
        X = FEATURESETS[fs].values[m]
        for mname, (model, grid) in ZOO.items():
            oof = nested_eval(X, y, model, grid, splits)
            rows.append(dict(target=t, featureset=fs, model=mname, n=len(y), **summarize(y, oof)))
            OOF[(t, fs, mname)] = oof.mean(axis=0)
    print(f"{t:8s} done ({time.time()-t0:.0f}s)")

bench = pd.DataFrame(rows)
nb = bench[bench.model != "Baseline(mean)"]
BEST = nb.loc[nb.groupby("target")["RMSE"].idxmin()].set_index("target")
base_rmse = bench[bench.model == "Baseline(mean)"].groupby("target")["RMSE"].mean()
BEST["RMSE_baseline"] = base_rmse
BEST["skill_vs_baseline"] = 1 - BEST["RMSE"] / BEST["RMSE_baseline"]
BEST["usable"] = BEST["Q2"] >= CFG["Q2_USABLE"]
display(BEST[["featureset", "model", "n", "RMSE", "RMSE_sd", "MAE", "Q2", "Q2_sd", "rho", "RMSE_baseline", "skill_vs_baseline", "usable"]].round(3))
weak = BEST.index[~BEST["usable"]].tolist()
if weak:
    print("WEAK property models (Q2 <", CFG["Q2_USABLE"], "):", weak, "-> add labelled data before trusting these predictions.")

In [ ]:
# ---------------- Figure 2: parity (out-of-fold) ----------------
def parity_grid():
    ts = list(BEST.index); ncol = 3; nrow = math.ceil(len(ts) / ncol)
    fig, axs = plt.subplots(nrow, ncol, figsize=(4.3 * ncol, 4.0 * nrow)); axs = np.atleast_1d(axs).ravel()
    for ax, t in zip(axs, ts):
        r = BEST.loc[t]; m = df[t].notna().values
        y = df.loc[m, t].values; p = OOF[(t, r.featureset, r.model)]
        ax.scatter(y, p, s=30, c=[ROLE_COLORS[x] for x in df.role[m]], edgecolor="white", lw=0.5)
        lo, hi = min(y.min(), p.min()), max(y.max(), p.max()); pad = 0.05 * (hi - lo + 1e-9)
        ax.plot([lo - pad, hi + pad], [lo - pad, hi + pad], "k--", lw=0.8)
        ax.set_title(f"{t}   Q2={r.Q2:.2f}  RMSE={r.RMSE:.3g}\n{r.model} / {r.featureset}", fontsize=8.5)
        ax.set_xlabel("experimental"); ax.set_ylabel("cross-validated prediction")
    for ax in axs[len(ts):]:
        ax.axis("off")
    plt.tight_layout(); savefig("fig2_parity_cv"); plt.show()
parity_grid()

# ---------------- Figure 3: model comparison + feature-set ablation ----------------
fig, axs = plt.subplots(1, 2, figsize=(13, 4.5))
sns.heatmap(bench.groupby(["target", "model"])["Q2"].max().unstack(), annot=True, fmt=".2f", cmap="viridis", vmin=-0.5, vmax=1, ax=axs[0], cbar_kws={"label": "CV Q2"})
axs[0].set_title("Model comparison (best feature set per cell)")
sns.heatmap(nb.groupby(["target", "featureset"])["Q2"].max().unstack(), annot=True, fmt=".2f", cmap="viridis", vmin=-0.5, vmax=1, ax=axs[1], cbar_kws={"label": "CV Q2"})
axs[1].set_title("Feature-set ablation (best model per cell)")
plt.tight_layout(); savefig("fig3_model_comparison"); plt.show()

## 5. Rigor checks reviewers ask for

1. **y-scrambling**: re-train on permuted labels; real Q2 must sit far above the scrambled distribution (empirical p-value).
2. **Cluster-split CV** (Butina clusters of Tanimoto similarity): whole structural families are held out, a much harder and more honest estimate of performance on *new* chemistry than random CV. Report both.
3. **Applicability domain** (OECD principle 3): leverage in PCA descriptor space (Williams plot, h* = 3(k+1)/n) plus maximum Tanimoto similarity to the training set. Predictions outside the domain are flagged and get *inflated uncertainty* in the screening.
4. **Mechanistic interpretation** (OECD principle 5): SHAP on a tree surrogate to show which descriptors drive each property and whether that is chemically sensible (e.g. TPSA/polarity for pi*, H-bond acceptor counts for beta).

In [ ]:
# ---------------- 5a. y-scrambling ----------------
rng = np.random.default_rng(SEED)
yscr_rows, yscr_dist = [], {}
for t in BEST.index:
    r = BEST.loc[t]; m = df[t].notna().values
    y = df.loc[m, t].values.astype(float); X = FEATURESETS[r.featureset].values[m]
    model, grid = ZOO[r.model]
    q2s = []
    for _ in range(N_PERM):
        yp = rng.permutation(y)
        o = nested_eval(X, yp, model, grid, kfold_repeats(len(y), 1))
        q2s.append(summarize(yp, o)["Q2"])
    yscr_dist[t] = q2s
    pval = (1 + sum(q >= r.Q2 for q in q2s)) / (1 + len(q2s))
    yscr_rows.append(dict(target=t, Q2_real=r.Q2, Q2_scrambled_mean=np.mean(q2s), Q2_scrambled_max=np.max(q2s), p_value=pval))
yscr = pd.DataFrame(yscr_rows).set_index("target")
display(yscr.round(3))

fig, ax = plt.subplots(figsize=(7.5, 4))
for i, t in enumerate(BEST.index):
    ax.scatter(yscr_dist[t], [i] * len(yscr_dist[t]), c="#999999", s=14, alpha=0.6)
    ax.scatter([BEST.loc[t, "Q2"]], [i], c="#D55E00", s=70, marker="D", zorder=3)
ax.set_yticks(range(len(BEST))); ax.set_yticklabels(BEST.index); ax.axvline(0, c="k", lw=0.6)
ax.set_xlabel("CV Q2"); ax.set_title("y-scrambling: real model (orange) vs permuted labels (grey)")
savefig("fig4_yscrambling"); plt.show()

In [ ]:
# ---------------- 5b. cluster-split CV (harder generalisation test) ----------------
def butina_groups(fps, cutoff=0.7):
    n = len(fps); dists = []
    for i in range(1, n):
        sims = DataStructs.BulkTanimotoSimilarity(fps[i], fps[:i])
        dists.extend([1 - s for s in sims])
    cl = Butina.ClusterData(dists, n, cutoff, isDistData=True)
    g = np.zeros(n, dtype=int)
    for ci, members in enumerate(cl):
        g[list(members)] = ci
    return g

groups_all = butina_groups(fps, 0.7)
print("Butina clusters:", len(np.unique(groups_all)), "for", len(fps), "solvents")

gcv = []
for t in BEST.index:
    r = BEST.loc[t]; m = df[t].notna().values
    g = groups_all[m]; ng = len(np.unique(g))
    if ng < 3:
        continue
    y = df.loc[m, t].values.astype(float); X = FEATURESETS[r.featureset].values[m]
    splits = [list(GroupKFold(n_splits=min(5, ng)).split(X, y, g))]
    o = nested_eval(X, y, *ZOO[r.model], splits)
    s = summarize(y, o)
    gcv.append(dict(target=t, Q2_random_cv=r.Q2, Q2_cluster_cv=s["Q2"], RMSE_random=r.RMSE, RMSE_cluster=s["RMSE"]))
gcv = pd.DataFrame(gcv).set_index("target")
display(gcv.round(3))

In [ ]:
# ---------------- 5c. final models, intervals, applicability domain ----------------
def leverage_ad(X_lab, X_all, k=5):
    imp = SimpleImputer(strategy="median").fit(X_lab)
    sc = StandardScaler().fit(imp.transform(X_lab))
    Zl, Za = sc.transform(imp.transform(X_lab)), sc.transform(imp.transform(X_all))
    pca = PCA(n_components=min(k, Zl.shape[0] - 1, Zl.shape[1]), random_state=SEED).fit(Zl)
    T, Tq = pca.transform(Zl), pca.transform(Za)
    G = np.linalg.pinv(T.T @ T)
    h = np.einsum("ij,jk,ik->i", Tq, G, Tq)
    return h, 3 * (T.shape[1] + 1) / T.shape[0]

FINAL, PRED, Q90, HSTAR = {}, {}, {}, {}
for t in BEST.index:
    r = BEST.loc[t]; m = df[t].notna().values
    y = df.loc[m, t].values.astype(float)
    Xa = FEATURESETS[r.featureset].values
    model, grid = ZOO[r.model]
    est, params = fit_final(Xa[m], y, model, grid)
    FINAL[t] = dict(estimator=est, params={k: (v if isinstance(v, (int, float, str)) else str(v)) for k, v in params.items()},
                    featureset=r.featureset, model=r.model)
    pred = est.predict(Xa)
    oof = OOF[(t, r.featureset, r.model)]
    Q90[t] = float(np.quantile(np.abs(y - oof), 0.90))          # 90% half-width from cross-validated residuals
    h, HSTAR[t] = leverage_ad(FEATURESETS["RDKit2D"].values[m], FEATURESETS["RDKit2D"].values)
    lab = np.where(m)[0]
    sim = np.array([max(DataStructs.BulkTanimotoSimilarity(fps[i], [fps[j] for j in lab if j != i])) for i in range(len(df))])
    PRED[t] = pd.DataFrame({"pred": pred, "lo": pred - Q90[t], "hi": pred + Q90[t], "lev": h, "sim_max": sim,
                            "in_AD": (h <= HSTAR[t]) & (sim >= CFG["SIM_MIN"])}, index=df["name"])
print("Final models fitted for:", list(FINAL))

# ---------------- Figure 5: Williams plots ----------------
show = [t for t in ["beta", "pi_star", "dP", "bp_C"] if t in PRED][:4]
if show:
    fig, axs = plt.subplots(1, len(show), figsize=(4.2 * len(show), 3.8)); axs = np.atleast_1d(axs)
    for ax, t in zip(axs, show):
        r = BEST.loc[t]; m = df[t].notna().values
        res = df.loc[m, t].values - OOF[(t, r.featureset, r.model)]
        z = res / (res.std() + 1e-12)
        ax.scatter(PRED[t].lev.values[m], z, s=28, c="#0072B2", edgecolor="white", lw=0.5)
        ax.axvline(HSTAR[t], c="#D55E00", ls="--"); ax.axhline(3, c="grey", ls=":"); ax.axhline(-3, c="grey", ls=":")
        ax.set_title(f"Williams plot: {t}", fontsize=9); ax.set_xlabel("leverage h"); ax.set_ylabel("standardised CV residual")
    plt.tight_layout(); savefig("fig5_williams"); plt.show()

In [ ]:
# ---------------- 5d. interpretation: SHAP on a tree surrogate ----------------
try:
    import shap
    shap_targets = [t for t in ["beta", "pi_star", "dP", "dH", "bp_C"] if t in BEST.index][:4]
    fig, axs = plt.subplots(1, max(len(shap_targets), 1), figsize=(4.6 * max(len(shap_targets), 1), 4)); axs = np.atleast_1d(axs)
    for ax, t in zip(axs, shap_targets):
        m = df[t].notna().values
        Xd = FEATURESETS["RDKit2D"]
        Xi = SimpleImputer(strategy="median").fit_transform(Xd.values[m])
        rf = RandomForestRegressor(n_estimators=N_TREES, random_state=SEED, n_jobs=-1).fit(Xi, df.loc[m, t].values)
        sv = shap.TreeExplainer(rf).shap_values(Xi)
        imp = pd.Series(np.abs(sv).mean(axis=0), index=Xd.columns).sort_values().tail(8)
        ax.barh(imp.index, imp.values, color="#0072B2"); ax.set_title(f"{t}: top descriptors (mean |SHAP|)", fontsize=9)
    plt.tight_layout(); savefig("fig6_shap"); plt.show()
except Exception as e:
    print("SHAP step skipped:", e)

## 6. Screening: property vectors, distance to DMF / NMP / DMAc

For every solvent and property we use the **experimental value if it exists, otherwise the QSPR prediction**. Predicted entries carry an uncertainty (90% half-width from cross-validated residuals; **doubled** when the solvent is outside the applicability domain), flagged `pred` or `pred*` (outside AD).

**Performance distance to the nearest reference solvent**

* Hansen distance: `Ra = sqrt(4(dD1-dD2)^2 + (dP1-dP2)^2 + (dH1-dH2)^2)`
* Kamlet-Taft distance: Euclidean in (alpha, beta, pi*)
* `d_perf = 0.5 * Ra/10 + 0.5 * KT/0.5` (fixed scales so that Monte-Carlo draws stay comparable)

**Other criteria:** hazard (CMR flag; unknown = data-gap penalty), bio-origin score (full 1 / potential 0.5 / fossil 0), process-window penalty (boiling window + minimum flash point; boiling point is used only if experimental or the bp model is usable, otherwise it is ignored rather than trusted).

In [ ]:
PROPS = ["dD", "dP", "dH", "alpha", "beta", "pi_star", "bp_C", "flash_C"]
SIXP = ["dD", "dP", "dH", "alpha", "beta", "pi_star"]
NONNEG = ["dD", "dP", "dH", "alpha", "beta", "pi_star"]

exp_tab = df.set_index("name")
VALM = pd.DataFrame(index=df["name"], columns=PROPS, dtype=float)
SD = pd.DataFrame(0.0, index=df["name"], columns=PROPS)
SRC = pd.DataFrame("missing", index=df["name"], columns=PROPS, dtype=object)
for t in PROPS:
    e = exp_tab[t]
    if t in PRED:
        p = PRED[t]
        VALM[t] = e.where(e.notna(), p["pred"])
        sdp = Q90[t] / 1.645
        SD[t] = np.where(e.notna(), 0.0, np.where(p["in_AD"], sdp, 2 * sdp))
        SRC[t] = np.where(e.notna(), "exp", np.where(p["in_AD"], "pred", "pred*"))
    else:
        VALM[t] = e
        SRC[t] = np.where(e.notna(), "exp", "missing")
for t in NONNEG:
    VALM[t] = VALM[t].clip(lower=0)

meta = exp_tab[["role", "bio_origin", "cmr"]].copy()
meta["hazard"] = meta["cmr"].map(lambda v: CFG["UNKNOWN_HAZARD"] if pd.isna(v) else float(v))
meta["bio"] = meta["bio_origin"].map({"full": 1.0, "potential": 0.5, "fossil": 0.0}).fillna(0.0)
# boiling point counts in the process-window criterion only if experimental, or if the bp model is usable (CV Q2 >= threshold)
BP_TRUST = (SRC["bp_C"] == "exp") | bool(("bp_C" in BEST.index) and BEST.loc["bp_C", "usable"])
print("Candidates whose boiling point is NOT trusted (predicted by a weak model) -> window penalty ignored:",
      [n for n in meta.index[(meta.role == "candidate")] if not BP_TRUST[n]])

def d_perf_from(V, R):
    # V: solvents x SIXP ; R: reference rows (same columns). returns DataFrame Ra_min, KT_min, d_perf
    ra, kt = [], []
    for ref in R.index:
        rr = R.loc[ref]
        ra.append(np.sqrt(4 * (V["dD"] - rr["dD"]) ** 2 + (V["dP"] - rr["dP"]) ** 2 + (V["dH"] - rr["dH"]) ** 2))
        kt.append(np.sqrt((V["alpha"] - rr["alpha"]) ** 2 + (V["beta"] - rr["beta"]) ** 2 + (V["pi_star"] - rr["pi_star"]) ** 2))
    out = pd.DataFrame({"Ra_min": pd.concat(ra, axis=1).min(axis=1), "KT_min": pd.concat(kt, axis=1).min(axis=1)})
    out["d_perf"] = 0.5 * out["Ra_min"] / 10 + 0.5 * out["KT_min"] / 0.5
    return out

def criteria(V, meta):
    pdist = d_perf_from(V[SIXP], V.loc[CFG["REFS"], SIXP])
    lo, hi = CFG["BP_WINDOW"]
    bp_term = ((lo - V["bp_C"]).clip(lower=0) / 50 + (V["bp_C"] - hi).clip(lower=0) / 50).where(BP_TRUST.reindex(V.index).fillna(False).astype(bool), 0.0)
    win = bp_term.fillna(0) + ((CFG["FLASH_MIN"] - V["flash_C"]).clip(lower=0) / 50).fillna(0)
    return pd.DataFrame({"d_perf": pdist["d_perf"], "Ra_min": pdist["Ra_min"], "KT_min": pdist["KT_min"],
                         "hazard": meta["hazard"], "bio": meta["bio"], "window": win.fillna(0)})

def topsis(M, w, benefit):
    X = M.values.astype(float)
    den = np.sqrt((X ** 2).sum(axis=0)); den[den == 0] = 1.0
    Vw = X / den * np.asarray(w)
    ideal = np.where(benefit, Vw.max(axis=0), Vw.min(axis=0))
    anti = np.where(benefit, Vw.min(axis=0), Vw.max(axis=0))
    dp = np.sqrt(((Vw - ideal) ** 2).sum(axis=1)); dm = np.sqrt(((Vw - anti) ** 2).sum(axis=1))
    return dm / (dp + dm + 1e-12)

def pareto_front(C):
    nd = np.ones(len(C), dtype=bool)
    for i in range(len(C)):
        for j in range(len(C)):
            if i != j and np.all(C[j] <= C[i]) and np.any(C[j] < C[i]):
                nd[i] = False; break
    return nd

CRIT = ["d_perf", "hazard", "bio", "window"]
BENEFIT = [False, False, True, False]
W = np.array([CFG["WEIGHTS"][k] for k in ("perf", "hazard", "bio", "window")])

cand_mask = (meta["role"] == "candidate") & (meta["cmr"].fillna(0) != 1)
excluded = meta.index[(meta["role"] == "candidate") & (meta["cmr"] == 1)].tolist()
if excluded:
    print("Candidates excluded for CMR classification:", excluded)
C0 = criteria(VALM, meta)
C0c = C0.loc[cand_mask[cand_mask].index].copy()
dropped = C0c.index[C0c["d_perf"].isna()].tolist()
if dropped:
    print("No complete property vector (too few labels to model) -> not ranked:", dropped)
C0c = C0c.dropna(subset=["d_perf"])

C0c["TOPSIS"] = topsis(C0c[CRIT], W, BENEFIT)
C0c["rank"] = C0c["TOPSIS"].rank(ascending=False, method="min").astype(int)
cost_mat = np.column_stack([C0c["d_perf"], C0c["hazard"], 1 - C0c["bio"], C0c["window"]])
C0c["pareto"] = pareto_front(cost_mat)
C0c["bp_trusted"] = BP_TRUST.reindex(C0c.index).fillna(False)
C0c["uses_prediction"] = SRC.loc[C0c.index, SIXP].ne("exp").any(axis=1)
C0c["outside_AD"] = SRC.loc[C0c.index, SIXP + ["bp_C"]].eq("pred*").any(axis=1)
C0c = C0c.join(VALM[["bp_C", "flash_C"]]).join(meta[["bio_origin"]])
print("Candidates ranked:", len(C0c))

In [ ]:
# ---------------- Figure 7: where do the candidates sit relative to DMF/NMP/DMAc? ----------------
fig, axs = plt.subplots(1, 2, figsize=(13, 5.2))
for ax, (xa, ya, xl, yl) in zip(axs, [("beta", "pi_star", "Kamlet-Taft beta (H-bond basicity)", "Kamlet-Taft pi* (dipolarity)"),
                                      ("dP", "dH", "Hansen dP (MPa$^{0.5}$)", "Hansen dH (MPa$^{0.5}$)")]):
    for role in ["conventional", "candidate", "reference"]:
        idx = meta.index[meta.role == role]
        pred_flag = SRC.loc[idx, [xa, ya]].ne("exp").any(axis=1).values
        ax.scatter(VALM.loc[idx, xa][~pred_flag], VALM.loc[idx, ya][~pred_flag], s=60, c=ROLE_COLORS[role],
                   edgecolor="white", lw=0.6, label=f"{role} (exp)")
        ax.scatter(VALM.loc[idx, xa][pred_flag], VALM.loc[idx, ya][pred_flag], s=60, facecolors="none",
                   edgecolor=ROLE_COLORS[role], lw=1.4, label=f"{role} (predicted)")
    for n in meta.index:
        if meta.loc[n, "role"] != "conventional":
            ax.annotate(n, (VALM.loc[n, xa], VALM.loc[n, ya]), fontsize=7, xytext=(4, 3), textcoords="offset points")
    ax.set_xlabel(xl); ax.set_ylabel(yl)
axs[0].legend(frameon=False, fontsize=7)
plt.tight_layout(); savefig("fig7_property_space"); plt.show()

## 7. Rank robustness: Monte-Carlo over weights **and** prediction uncertainty

A single ranking hides two kinds of doubt: (i) the criteria weights are a judgement call, and (ii) predicted properties are uncertain. Each draw below (a) samples weights from a Dirichlet distribution centred on the base weights, (b) perturbs every *predicted* property with Gaussian noise whose SD comes from cross-validated residuals (x2 outside the AD), then recomputes distances and the TOPSIS ranking. We report median rank, inter-quartile range, and **P(top-3)** / **P(top-5)**. This is what the abstract should quote.

In [ ]:
def draw_values(rng):
    V = VALM + rng.normal(size=VALM.shape) * SD.values
    for t in NONNEG:
        V[t] = V[t].clip(lower=0)
    return V

rng = np.random.default_rng(SEED)
cand_names = C0c.index.tolist()
ranks = np.full((CFG["N_MC"], len(cand_names)), np.nan)
for b in range(CFG["N_MC"]):
    w = rng.dirichlet(W * 20)
    Cb = criteria(draw_values(rng), meta).loc[cand_names]
    ok = Cb["d_perf"].notna().values
    s = np.full(len(cand_names), np.nan)
    s[ok] = topsis(Cb.loc[ok, CRIT], w, BENEFIT)
    ranks[b] = (-s).argsort().argsort() + 1

ranking = C0c.copy()
ranking["rank_median"] = np.nanmedian(ranks, axis=0)
ranking["rank_q25"] = np.nanpercentile(ranks, 25, axis=0)
ranking["rank_q75"] = np.nanpercentile(ranks, 75, axis=0)
ranking["P_top3"] = (ranks <= 3).mean(axis=0)
ranking["P_top5"] = (ranks <= 5).mean(axis=0)
ranking = ranking.sort_values(["rank", "rank_median"]).reset_index().rename(columns={"index": "name"})
show_cols = ["name", "bio_origin", "bp_C", "flash_C", "Ra_min", "KT_min", "d_perf", "hazard", "TOPSIS", "rank",
             "rank_median", "P_top3", "P_top5", "pareto", "uses_prediction", "outside_AD", "bp_trusted"]
display(ranking[show_cols].round(3))

In [ ]:
# ---------------- ranking sensitivity: what if the (unverified) hazard flags are ignored? ----------------
W2 = W.copy(); W2[CRIT.index("hazard")] = 0.0; W2 = W2 / W2.sum()
tmp = C0c.copy()
tmp["rank_no_hazard"] = pd.Series(topsis(tmp[CRIT], W2, BENEFIT), index=tmp.index).rank(ascending=False, method="min").astype(int)
ranking = ranking.merge(tmp[["rank_no_hazard"]], left_on="name", right_index=True, how="left")
print("Rank with and without the hazard criterion (large changes = result depends on hazard flags; verify cmr first):")
display(ranking[["name", "hazard", "rank", "rank_no_hazard", "P_top3", "bp_trusted", "uses_prediction", "outside_AD"]].head(15).round(3))

In [ ]:
# ---------------- Figure 8: ranking with robustness; Figure 9: Pareto view ----------------
top = ranking.head(15).iloc[::-1]
fig, axs = plt.subplots(1, 2, figsize=(13, 5.5))
colors = top["bio_origin"].map({"full": "#009E73", "potential": "#E69F00", "fossil": "#7F7F7F"})
axs[0].hlines(top["name"], top["rank_q25"], top["rank_q75"], color=colors, lw=4, alpha=0.7)
axs[0].scatter(top["rank_median"], top["name"], c=colors, s=55, edgecolor="k", zorder=3)
axs[0].set_xlabel("rank over Monte-Carlo draws (median, IQR)  -  lower is better"); axs[0].set_title("Rank robustness (green = fully bio-based)")
sust = 0.5 * ranking["hazard"] + 0.5 * (1 - ranking["bio"].fillna(0)) if "bio" in ranking else 0.5 * ranking["hazard"]
axs[1].scatter(ranking["d_perf"], sust, s=45, c=np.where(ranking["pareto"], "#D55E00", "#BBBBBB"), edgecolor="white")
for _, r_ in ranking.iterrows():
    if r_["pareto"] or r_["rank"] <= 5:
        axs[1].annotate(r_["name"], (r_["d_perf"], 0.5 * r_["hazard"] + 0.5 * (1 - r_["bio"])), fontsize=7, xytext=(4, 3), textcoords="offset points")
axs[1].set_xlabel("distance to DMF/NMP/DMAc (d_perf)  -  lower is better"); axs[1].set_ylabel("sustainability burden (hazard + non-bio)")
axs[1].set_title("Pareto-optimal candidates (orange)")
plt.tight_layout(); savefig("fig8_9_ranking_pareto"); plt.show()

## 8. Blind retrospective validation

Do the QSPR-derived property vectors, which for each solvent come from a model that **never saw that solvent** (out-of-fold predictions), still place literature-known DMF/NMP replacements close to the references?

* Spearman correlation between `d_perf` from OOF predictions and from experimental values.
* ROC-AUC for separating known replacements (`POSITIVES`, edit this list with citations) from other solvents, using `-d_perf` as the score. Only solvents with complete experimental *and* OOF vectors can be evaluated; with a small library the number of positives is tiny, so treat this as a protocol, not yet a result.

In [ ]:
POSITIVES = ["Cyrene", "GVL", "DMSO", "Propylene carbonate", "NBP", "PolarClean", "Dimethyl isosorbide"]   # EDIT + CITE

Vexp = exp_tab[SIXP]
Voof = pd.DataFrame(np.nan, index=df["name"], columns=SIXP)
for t in SIXP:
    if t in BEST.index:
        r = BEST.loc[t]; m = df[t].notna().values
        Voof.loc[df["name"][m].values, t] = OOF[(t, r.featureset, r.model)]

Rexp = Vexp.loc[CFG["REFS"]]
ok = Vexp.notna().all(axis=1) & Voof.notna().all(axis=1) & ~Vexp.index.isin(CFG["REFS"])
if ok.sum() >= 5:
    d_exp = d_perf_from(Vexp[ok], Rexp)["d_perf"]
    d_oof = d_perf_from(Voof[ok], Rexp)["d_perf"]
    rho, pval = stats.spearmanr(d_exp, d_oof)
    y_true = d_exp.index.isin(POSITIVES)
    print(f"Solvents evaluated: {ok.sum()} | known replacements among them: {int(y_true.sum())}")
    print(f"Spearman(d_perf_exp, d_perf_blind_QSPR) = {rho:.2f} (p={pval:.3g})")
    if 0 < y_true.sum() < len(y_true):
        print(f"ROC-AUC, experimental vectors : {roc_auc_score(y_true, -d_exp.values):.2f}")
        print(f"ROC-AUC, blind QSPR vectors   : {roc_auc_score(y_true, -d_oof.values):.2f}")
        print("Positives evaluated:", d_exp.index[y_true].tolist())
    fig, ax = plt.subplots(figsize=(4.8, 4.5))
    ax.scatter(d_exp, d_oof, c=np.where(y_true, "#009E73", "#999999"), s=45, edgecolor="white")
    lim = [0, max(d_exp.max(), d_oof.max()) * 1.05]; ax.plot(lim, lim, "k--", lw=0.8)
    ax.set_xlabel("d_perf (experimental)"); ax.set_ylabel("d_perf (blind QSPR)"); ax.set_title("Retrospective validation")
    savefig("fig10_retrospective"); plt.show()
else:
    print("Not enough solvents with complete experimental + OOF vectors for the retrospective test. Add labelled data.")

## 9. Extension: binary-blend design (novelty hook)

Single bio-solvents rarely match DMF/NMP exactly, but **blends** can. Here Hansen parameters are mixed by volume fraction (standard rule) and Kamlet-Taft parameters are mixed linearly, which is only a first approximation because of preferential solvation. Treat the output as a *shortlist for experiments*, and verify with measured solvatochromic data for the top blends.

In [ ]:
extra_pool = [s for s in ["Ethanol", "Acetone", "Ethyl acetate", "DMSO", "Dimethyl carbonate"] if s in VALM.index]
pool = list(dict.fromkeys(ranking["name"].head(12).tolist() + extra_pool))
Rref = VALM.loc[CFG["REFS"], SIXP]
brow = []
for a, b in itertools.combinations(pool, 2):
    for f in np.arange(0.1, 1.0, 0.1):
        v = f * VALM.loc[a, SIXP] + (1 - f) * VALM.loc[b, SIXP]
        if v.isna().any():
            continue
        d = d_perf_from(pd.DataFrame([v.values], index=["blend"], columns=SIXP), Rref).iloc[0]
        brow.append(dict(blend=f"{f:.1f} {a} + {1-f:.1f} {b}", d_perf=d["d_perf"], Ra_min=d["Ra_min"], KT_min=d["KT_min"],
                         hazard=f * meta.loc[a, "hazard"] + (1 - f) * meta.loc[b, "hazard"],
                         bio=f * meta.loc[a, "bio"] + (1 - f) * meta.loc[b, "bio"]))
blends = pd.DataFrame(brow).sort_values("d_perf").reset_index(drop=True)
display(blends.head(15).round(3))

In [ ]:
# ---------------- ORCA input generator (for QM descriptors / validation of the top candidates) ----------------
def write_orca_inputs(names, outdir="orca_inputs", method="r2SCAN-3c", extra="TightSCF", nprocs=4, maxcore=3000, n_confs=30):
    out = Path(outdir); out.mkdir(exist_ok=True)
    lookup = df.set_index("name")["smiles_can"]
    written = []
    for n in names:
        try:
            mol = Chem.AddHs(Chem.MolFromSmiles(lookup[n]))
            cids = list(AllChem.EmbedMultipleConfs(mol, numConfs=n_confs, randomSeed=SEED))
            res = AllChem.MMFFOptimizeMoleculeConfs(mol, maxIters=5000)
            best = cids[int(np.argmin([x[1] for x in res]))]
            slug = "".join(ch if ch.isalnum() else "_" for ch in n)
            (out / f"{slug}.xyz").write_text(Chem.MolToXYZBlock(mol, confId=best))
            (out / f"{slug}.inp").write_text(
                f"! {method} Opt Freq {extra}\n%pal nprocs {nprocs} end\n%maxcore {maxcore}\n* xyzfile 0 1 {slug}.xyz\n")
            written.append(slug)
        except Exception as e:
            print("skipped", n, "->", e)
    return written

def parse_orca_out(path):
    txt = Path(path).read_text(errors="ignore")
    dip = re.findall(r"Magnitude \(Debye\)\s*:\s*([-\d.]+)", txt)
    en = re.findall(r"FINAL SINGLE POINT ENERGY\s+([-\d.]+)", txt)
    return dict(dipole_D=float(dip[-1]) if dip else np.nan, E_h=float(en[-1]) if en else np.nan)

top_names = ranking["name"].head(10).tolist() + CFG["REFS"]
print("ORCA inputs written for:", write_orca_inputs(top_names))
# Workflow: run ORCA locally -> parse each .out with parse_orca_out() -> save a CSV with columns
# name, dipole_D, E_h, ... as qm_descriptors.csv -> upload it to Colab -> re-run from Section 3.
# The notebook then adds the feature set 'RDKit2D+QM' automatically.

## 9b. Paper-strengthening extras (automatic; results go into the output zip)
Everything below runs by itself after the main analysis: weight sensitivity, unknown-hazard scenarios, literature hold-out (Cyrene), manuscript tables, an optional internet hazard screen (PubChem) and an optional reference finder (Crossref). Cells that need internet skip themselves safely if it is unavailable.

In [ ]:
# ---------- 9b-1. weight sensitivity + unknown-hazard scenarios (uses the ranking above) ----------
import itertools
from scipy.stats import kendalltau
_CR = ["d_perf", "hazard", "bio", "window"]; _BEN = [False, False, True, False]; _W0 = np.array([0.45, 0.25, 0.20, 0.10])
_R = ranking.copy().reset_index(drop=True)
chk = np.abs(topsis(_R[_CR], _W0, _BEN) - _R["TOPSIS"]).max(); print("TOPSIS reproduction check, max diff:", float(chk))
rows = []
for w in itertools.product(np.arange(0.05, 0.8, 0.05), repeat=3):
    if sum(w) >= 0.96: continue
    w4 = np.array([*w, 1 - sum(w)])
    rk = pd.Series(topsis(_R[_CR], w4, _BEN), index=_R["name"]).rank(ascending=False, method="min")
    rows.append(rk)
SENS = pd.DataFrame(rows); top8 = _R.sort_values("rank")["name"].head(8).tolist()
wsum = pd.DataFrame({"solvent": top8, "share_rank1": [(SENS[n] == 1).mean() for n in top8], "share_top3": [(SENS[n] <= 3).mean() for n in top8],
                     "share_top5": [(SENS[n] <= 5).mean() for n in top8], "worst_rank": [int(SENS[n].max()) for n in top8]}).round(3)
display(wsum); wsum.to_csv(OUT / "weight_sensitivity_summary.csv", index=False); print(len(SENS), "weight combinations")
fig, ax = plt.subplots(figsize=(6.5, 3.6)); xx = np.arange(len(top8))
ax.bar(xx - 0.2, wsum.share_rank1, 0.4, label="rank 1"); ax.bar(xx + 0.2, wsum.share_top3, 0.4, label="top 3")
ax.set_xticks(xx); ax.set_xticklabels(top8, rotation=35, ha="right"); ax.set_ylabel("share of weight combinations"); ax.legend(); plt.tight_layout()
savefig("fig11_weight_sensitivity"); plt.show()
unk = _R["hazard"] == 0.5
def _scen(v):
    M = _R[_CR].copy(); M.loc[unk, "hazard"] = v
    return pd.Series(topsis(M, _W0, _BEN), index=_R["name"]).rank(ascending=False, method="min").astype(int)
UNK = pd.DataFrame({"solvent": _R["name"], "hazard_known": ~unk.values, "rank_baseline": _R["rank"].values,
                    "rank_if_unknown_safe": _scen(0.0).values, "rank_if_unknown_hazardous": _scen(1.0).values}).sort_values("rank_baseline")
display(UNK.head(15)); UNK.to_csv(OUT / "unknown_hazard_scenarios.csv", index=False)

In [ ]:
# ---------- 9b-2. literature hold-out check for Cyrene (Cyrene must have no labels in the dataset) ----------
HOLD = pd.DataFrame([
    ("Cyrene", "alpha", 0.00, "Sherwood et al., Chem. Commun. 2014, 50, 9650 (SI)"),
    ("Cyrene", "beta", 0.61, "Sherwood et al., Chem. Commun. 2014, 50, 9650 (SI)"),
    ("Cyrene", "pi_star", 0.93, "Sherwood et al., Chem. Commun. 2014, 50, 9650 (SI)"),
    ("Cyrene", "dD", 18.8, "Hansen value cited in Sherwood 2014 (computed, not measured)"),
    ("Cyrene", "dP", 10.6, "Hansen value cited in Sherwood 2014 (computed, not measured)"),
    ("Cyrene", "dH", 6.9, "Hansen value cited in Sherwood 2014 (computed, not measured)")], columns=["name", "property", "literature", "source"])
# add more rows here (name must match the dataset; remove that solvent's value from the dataset first so it is a true hold-out)
res = []
for _, h in HOLD.iterrows():
    if h["property"] not in PRED or h["name"] not in PRED[h["property"]].index: continue
    lab = df.set_index("name").loc[h["name"], h["property"]] if h["property"] in df.columns else np.nan
    p = PRED[h["property"]].loc[h["name"]]
    res.append(dict(name=h["name"], property=h["property"], literature=h["literature"], pred=p["pred"], lo=p["lo"], hi=p["hi"],
                    training_label_present=bool(pd.notna(lab)), abs_error=abs(p["pred"] - h["literature"]), inside_90=bool(p["lo"] <= h["literature"] <= p["hi"]), source=h["source"]))
HR = pd.DataFrame(res); display(HR.round(3))
if len(HR): print("Coverage of 90% intervals:", round(HR.inside_90.mean(), 2), "| n =", len(HR), "| any training label present (invalid hold-out)?", bool(HR.training_label_present.any()))
HR.to_csv(OUT / "holdout_cyrene_results.csv", index=False)

In [ ]:
# ---------- 9b-3. manuscript tables (LaTeX + CSV) ----------
t2 = ranking.sort_values("rank").head(10)[["rank", "name", "TOPSIS", "P_top3", "rank_no_hazard", "outside_AD"]].copy()
t2["outside_AD"] = t2["outside_AD"].map({True: "Yes", False: "No"})
t2.columns = ["Rank", "Solvent", "TOPSIS", "P(top-3)", "Rank w/o hazard", "Outside AD"]
t2.round(3).to_csv(OUT / "table2_ranking.csv", index=False)
body = "\n".join(f"{int(r['Rank'])} & {r['Solvent']} & {r['TOPSIS']:.3f} & {r['P(top-3)']:.2f} & {int(r['Rank w/o hazard'])} & {r['Outside AD']} \\\\" for _, r in t2.iterrows())
tex = ("\\begin{table}[ht]\n\\centering\n\\caption{Top candidates ranked by TOPSIS; P(top-3) from Monte-Carlo draws; AD = applicability domain.}\n"
       "\\label{tab:ranking}\n\\begin{tabular}{rlcccc}\n\\toprule\nRank & Solvent & TOPSIS & P(top-3) & Rank w/o hazard & Outside AD \\\\\n\\midrule\n" + body + "\n\\bottomrule\n\\end{tabular}\n\\end{table}\n")
(OUT / "table2_ranking.tex").write_text(tex)
best = BEST[["featureset", "model", "n", "Q2", "RMSE", "usable"]].round(3)
best.to_csv(OUT / "table1_models.csv"); print(tex); display(best)

In [ ]:
# ---------- 9b-4. OPTIONAL internet hazard screen (PubChem GHS aggregates; NOT the harmonised ECHA classification) ----------
import re, time
try:
    import requests
    CMRX = re.compile(r"H3[45][01]|H36[01]")
    def _h(o, out):
        if isinstance(o, dict): [ _h(v, out) for v in o.values() ]
        elif isinstance(o, list): [ _h(v, out) for v in o ]
        elif isinstance(o, str): out.update(re.findall(r"H\d{3}[A-Za-z]*", o))
    todo = df[(df["role"] == "candidate") & (~df["source"].astype(str).str.contains("VERIFIED"))]["name"].tolist()
    rows = []
    for n in todo:
        try:
            cid = requests.get(f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/name/{requests.utils.quote(n)}/cids/JSON", timeout=25).json()["IdentifierList"]["CID"][0]
            v = requests.get(f"https://pubchem.ncbi.nlm.nih.gov/rest/pug_view/data/compound/{cid}/JSON?heading=GHS+Classification", timeout=40)
            codes = set(); _h(v.json(), codes) if v.status_code == 200 else None
            rows.append(dict(name=n, cid=cid, hcodes=" ".join(sorted(codes)), screen=("CMR code present" if any(CMRX.match(c) for c in codes) else ("no CMR code listed" if codes else "no data"))))
        except Exception as e:
            rows.append(dict(name=n, cid=None, hcodes="", screen="lookup failed"))
        time.sleep(0.3)
    PH = pd.DataFrame(rows); display(PH); PH.to_csv(OUT / "pubchem_ghs_screen_candidates.csv", index=False)
    print("Screening aid only: confirm important entries in the ECHA C&L Inventory before changing any cmr value.")
except Exception as e:
    print("Skipped (no internet or requests unavailable):", str(e)[:80])

In [ ]:
# ---------- 9b-5. OPTIONAL reference finder (Crossref). Prints top-3 matches per topic; YOU must pick the right one ----------
REFQ = {
 "Cyrene as dipolar aprotic solvent (Sherwood 2014)": "Dihydrolevoglucosenone (Cyrene) as a bio-based alternative for dipolar aprotic solvents",
 "Kamlet-Taft solvent parameters of green solvents (Jessop 2012)": "Jessop Kamlet-Taft solvent parameters green solvents Green Chemistry 2012",
 "CHEM21 solvent selection guide": "CHEM21 selection guide of classical- and less classical-solvents",
 "GSK solvent selection guide": "Expanding GSK's solvent selection guide",
 "GVL as sustainable liquid (Horvath 2008)": "gamma-Valerolactone a sustainable liquid for energy and carbon-based chemicals",
 "Butina clustering": "Unsupervised data base clustering based on Daylight's fingerprint and Tanimoto similarity",
 "SHAP (Lundberg and Lee)": "A unified approach to interpreting model predictions",
 "scikit-learn": "Scikit-learn: Machine learning in Python",
 "Replacement of DMF/NMP (review)": "replacement of dipolar aprotic solvents DMF NMP greener alternatives review",
 "Hansen solubility parameters of bio-based solvents": "Hansen solubility parameters bio-based solvents replacement of NMP DMF",
}
try:
    import requests
    ref_rows = []
    for topic, q in REFQ.items():
        j = requests.get("https://api.crossref.org/works", params={"query.bibliographic": q, "rows": 3}, timeout=30).json()["message"]["items"]
        for k, it in enumerate(j):
            au = "; ".join(f"{a.get('family','')} {a.get('given','')[:1]}" for a in it.get("author", [])[:3])
            ref_rows.append(dict(topic=topic, rank=k + 1, title=(it.get("title") or [""])[0], authors=au, journal=(it.get("container-title") or [""])[0],
                                 year=(it.get("issued", {}).get("date-parts") or [[None]])[0][0], doi=it.get("DOI", "")))
    REF = pd.DataFrame(ref_rows); display(REF[["topic", "rank", "title", "year", "doi"]])
    REF.to_csv(OUT / "reference_candidates_VERIFY.csv", index=False)
    print("Open each DOI and confirm title/authors/year before citing. Hansen's handbook and TOPSIS (Hwang & Yoon 1981) are books: cite them manually.")
except Exception as e:
    print("Skipped (no internet):", str(e)[:80])

In [ ]:
# ---------- 9b-6. one-page results summary for the manuscript ----------
g = ranking.set_index("name")
lines = ["# Results summary (auto-generated)", f"Solvents: {len(df)} | FAST={CFG['FAST']} | seed {SEED}", "", "## Models (nested CV)"]
for t in BEST.index:
    lines.append(f"- {t}: {BEST.loc[t,'model']} / {BEST.loc[t,'featureset']}  Q2={BEST.loc[t,'Q2']:.3f}  RMSE={BEST.loc[t,'RMSE']:.3f}  n={int(BEST.loc[t,'n'])}  usable={bool(BEST.loc[t,'usable'])}")
lines += ["", "## Top ranking"] + [f"- {i+1}. {n}: TOPSIS {g.loc[n,'TOPSIS']:.3f}, P(top-3) {g.loc[n,'P_top3']:.2f}, rank without hazard {int(g.loc[n,'rank_no_hazard'])}" for i, n in enumerate(ranking.sort_values('rank')['name'].head(8))]
lines += ["", f"GVL ranks first in {wsum.set_index('solvent').loc[top8[0],'share_rank1']*100:.0f}% of {len(SENS)} weight combinations (top solvent: {top8[0]})."]
(OUT / "results_summary.md").write_text("\n".join(lines)); print("\n".join(lines))

## 10. Export for the manuscript / SI / Zenodo

In [ ]:
bench.to_csv(OUT / "qspr_benchmark_all.csv", index=False)
BEST.to_csv(OUT / "qspr_best_models.csv")
yscr.to_csv(OUT / "y_scrambling.csv")
gcv.to_csv(OUT / "cluster_cv.csv")
ranking.to_csv(OUT / "candidate_ranking.csv", index=False)
blends.to_csv(OUT / "blend_shortlist.csv", index=False)
long = []
for t, p in PRED.items():
    long.append(p.reset_index().assign(property=t))
pd.concat(long).to_csv(OUT / "predictions_with_intervals_and_AD.csv", index=False)
df.drop(columns=["inchikey"]).to_csv(OUT / "curated_dataset.csv", index=False)
joblib.dump({t: v for t, v in FINAL.items()}, OUT / "final_models.joblib")

import sklearn, rdkit
card = dict(
    title="QSPR/ML screening of bio-based replacements for DMF/NMP/DMAc",
    created=time.strftime("%Y-%m-%d"), seed=SEED, config={k: (v if not isinstance(v, tuple) else list(v)) for k, v in CFG.items()},
    n_solvents=int(len(df)), models={t: dict(model=v["model"], featureset=v["featureset"], params=v["params"]) for t, v in FINAL.items()},
    cv_q2={t: float(BEST.loc[t, "Q2"]) for t in BEST.index}, cv_rmse={t: float(BEST.loc[t, "RMSE"]) for t in BEST.index},
    versions=dict(python=platform.python_version(), numpy=np.__version__, pandas=pd.__version__, sklearn=sklearn.__version__, rdkit=rdkit.__version__),
    intended_use="rank candidate solvents for experimental follow-up; NOT a substitute for toxicological assessment",
)
(OUT / "model_card.json").write_text(json.dumps(card, indent=2, default=str))

shutil.make_archive("green_solvent_outputs", "zip", OUT)
print("Saved:", sorted(p.name for p in OUT.iterdir()))
try:
    from google.colab import files
    files.download("green_solvent_outputs.zip")
except Exception:
    pass

## 11. From notebook to manuscript

**Working title options**
1. *Uncertainty-aware machine-learning screening of bio-based solvents as replacements for DMF, NMP and DMAc*
2. *Beyond Hansen distance: multi-objective QSPR-guided design of green dipolar aprotic solvents and blends*

**Realistic target tiers** (choose by how much new experimental data you add): *Green Chemistry*, *ACS Sustainable Chemistry & Engineering*, *Journal of Chemical Information and Modeling*, *Digital Discovery*; solid fall-backs: *Journal of Cheminformatics*, *Journal of Cleaner Production*, *Journal of Molecular Liquids*.

**Figure plan:** (1) graphical abstract / workflow, (2) chemical space, (3) benchmark + ablation, (4) parity + cluster-CV, (5) AD (Williams) + SHAP, (6) property space with references, (7) robust ranking + Pareto, (8) retrospective validation, (9) blends, (10) experimental validation.

**Must-do before submission**
- [ ] Replace the seed table with a curated dataset (>=150 solvents), cite the source of every value, dedupe conflicting literature values (document your rule).
- [ ] Verify CMR flags against the ECHA C&L Inventory / SDS; document how unknown hazards are treated.
- [ ] Final run with `FAST=False`; fix the random seed; archive code+data on GitHub + Zenodo (DOI in the paper).
- [ ] Report cluster-CV numbers, y-scrambling p-values, AD coverage, and the weak models honestly.
- [ ] Add at least one experimental validation (Kamlet-Taft by solvatochromic dyes for the top 3-5; or polymer/membrane solubility; or an amide coupling / SNAr benchmark vs DMF) and show whether the predicted ranking is reproduced.
- [ ] OECD QSAR validation-principles table in the SI; model card from `outputs/model_card.json`.
- [ ] Discuss limitations: small N, linear mixing for blends, hazard data gaps, bio-origin is not life-cycle assessment.